# Preparing Thermal States of Frustrated Quantum Spin Systems

This challenge notebook is based on *Preparing thermal states of frustrated quantum spin <span style="color:darkgreen">systems</span> using 139 qubits* ([arXiv:2605.26245](https://arxiv.org/abs/2605.26245)).

The central question is:

> **Can engineered dissipation cool a frustrated quantum magnet toward a thermal-like steady state, and does the same mechanism remain visible when we scale from a classically simulable regime to a 50+ qubit quantum experiment?**

We will move through three regimes:

$$
\boxed{\text{ideal statevector simulation}}
\;\longrightarrow\;
\boxed{\text{24-qubit hardware benchmark}}
\;\longrightarrow\;
\boxed{\text{utility-scale }\sim 67\text{-qubit experiment}}.
$$

**Requirements.** This notebook requires Jupyter, Python 3.14 and an IBM Quantum account. Install the packages listed in the challenge's `requirements.txt`.

**Team name.** Answers are only graded for a team: set `TEAM_NAME` in the cell below and run it before any grading cell.

In [ ]:
from qc_grader.challenges.qdc_2026 import (
    join_team,
    grade_lab_thermal_states_ex1_2,
    grade_lab_thermal_states_ex2_6,
    grade_lab_thermal_states_ex2_7,
    grade_lab_thermal_states_ex2_8a,
    grade_lab_thermal_states_ex2_8b,
    grade_lab_thermal_states_ex2_8c,
    grade_lab_thermal_states_ex2_8e,
    grade_lab_thermal_states_ex2_8f,
    grade_lab_thermal_states_ex2_8_concepts,
    grade_lab_thermal_states_ex2_8d_g,
    grade_lab_thermal_states_ex2_11,
    grade_lab_thermal_states_ex2_14,
    grade_lab_thermal_states_ex3_4,
    grade_lab_thermal_states_ex3_5,
    grade_lab_thermal_states_ex3_6,
    grade_lab_thermal_states_ex4_1,
    grade_lab_thermal_states_ex4_2,
    grade_lab_thermal_states_ex4_7,
)
# from qc_grader.challenges.qdc_2026 import check_progress

TEAM_NAME = ""  # TODO: your team's name

if TEAM_NAME:
    join_team(TEAM_NAME)
else:
    print("Set TEAM_NAME to your team's name and run this cell again: answers are only graded for a team.")

# **PART 0: Background**

### The frustrated <span style="color:darkgreen">System</span> we want to thermalize

We will focus on the antiferromagnetic Ising Hamiltonian, which takes the following form: $$H_{\text{AFIM}} = \sum_{\langle i_S, j_S \rangle} Z_{i_S} Z_{j_S} + \sum_{i_S} (g_x X_{i_S} + g_z Z_{i_S})$$

We will use this Hamiltonian as our <span style="color:darkgreen">System</span> Hamiltonian ($H_S = H_{\text{AFIM}}$). In particular, we will focus on the case where the pairs $\langle i_S, j_S \rangle$ are nearest neighbors on the kagome lattice. 

The positive nearest-neighbor coupling favors antiparallel alignment of neighboring Ising spins. On an isolated triangle, however, this preference cannot be satisfied on all three bonds simultaneously, i.e., if two spins are antiparallel, the third spin must be parallel to at least one of them. This local incompatibility is the origin of **geometric frustration**. Because the kagome lattice consists of corner-sharing triangles, these frustrated local constraints are repeated and coupled throughout the <span style="color:darkgreen">System</span>, producing a highly degenerate manifold of competing spin configurations. Thus, frustration is not a consequence of disorder or competing interaction strengths, but arises directly from the geometry of the lattice and the antiferromagnetic interaction. 

Here, we work with $g_x=0.5, g_z=2.0$. At low temperature, the chosen parameters place the model in the frustrated kagome-ice regime studied in the reference paper. 

### What is a Gibbs sampler?

For the <span style="color:darkgreen">System</span> Hamiltonian $H_S$, thermal equilibrium at inverse temperature $\beta=1/(k_B T)$ is described by the Gibbs state

$$
\rho_\beta = \frac{e^{-\beta H_S}}{Z},\qquad
Z=\operatorname{Tr}(e^{-\beta H_S}).
$$

Two limits are useful to keep in mind:

- $\beta=0$: Infinite temperature, $\rho_0=I/2^{N_S}$ so all <span style="color:darkgreen">System</span> states are equally populated.
- $\beta\rightarrow\infty$: The state becomes concentrated in the lowest-energy sector, or the ground-state subspace when the ground state is degenerate.

Preparing $\rho_\beta$ is harder than preparing an ordinary wavefunction because a finite-temperature Gibbs state is generally **mixed**. The dissipative Gibbs sampler avoids directly implementing the non-unitary operator $e^{-\beta H_S}$. Instead, it repeatedly couples the <span style="color:darkgreen">System</span> to a small engineered <span style="color:red">Environment</span> and then resets that <span style="color:red">Environment</span>. One reset cycle defines a quantum channel

$$
\rho_S \mapsto \Phi(\rho_S),
$$

whose fixed point is designed to approximate the target Gibbs state.

The channel is designed so that, after many cycles,

$$
\Phi^n(\rho_S)\longrightarrow\rho_\beta,
$$

or, in an approximate implementation, to a state close to $\rho_\beta$.

The central physical principle is **detailed balance**. Suppose the <span style="color:darkgreen">System</span> can undergo transitions between energy eigenstates $i$ and $j$, with energies $E_i$ and $E_j$. At thermal equilibrium, the transition rates satisfy

$$
\frac{\Gamma_{i\rightarrow j}}
     {\Gamma_{j\rightarrow i}}
=
e^{-\beta(E_j-E_i)}.
$$

Thus, transitions that increase the <span style="color:darkgreen">System's</span> energy are suppressed relative to the corresponding energy-lowering transitions by the appropriate Boltzmann factor. This asymmetry drives the <span style="color:darkgreen">System</span> toward the thermal population:

$$
p_i\propto e^{-\beta E_i}.
$$

In this picture, the <span style="color:red">Environment</span> acts as a **thermalizer**. It exchanges energy with the <span style="color:darkgreen">System</span> while the reset operation removes the <span style="color:red">Environment's</span> accumulated entropy and returns it to its prescribed state. When the resulting channel satisfies the appropriate detailed-balance condition, the Gibbs state is a fixed point

$$
\Phi(\rho_\beta)=\rho_\beta,
$$

and repeated application of the channel drives the <span style="color:darkgreen">System</span> toward thermal equilibrium.

A useful way to summarize the idea is that the Gibbs sampler does not prepare the Gibbs state in a single step. Instead, it engineers a dynamical process whose long-time fixed point is the desired Gibbs state.


### Engineered <span style="color:red">Environment</span> and jump operators

We use a classical <span style="color:red">Environment</span> Hamiltonian

$$
H_E = -\sum_{i_E}\frac{\omega_{i_E}}{2}Z_{i_E},
$$

where the Bohr frequencies $\omega_{i_E}\in(0,\omega_{\max}]$ are resampled from a uniform distribution during every reset cycle.

A bath qubit $i_E$ with spacing $\omega_{i_E}$ is initially thermally excited with probability:

$$
P(|i_E(0)\rangle = |1\rangle)=\frac{1}{1+e^{\beta\omega_{i_E}}}.
$$

The <span style="color:darkgreen">System</span> and <span style="color:red">Environment</span> interact through

$$
H_{SE}=\sum_{\langle i_S,i_E\rangle} O_{i_S}\otimes X_{i_E},
$$

where the jump operators are sampled from

$$O_{i_S}\in\{\pm X_{i_S},\pm Y_{i_S},\pm Z_{i_S}\}.$$

These operators induce transitions between eigenstates of $H_S$. Randomizing the Bohr frequencies allows the <span style="color:red">Environment</span> to probe different many-body energy differences without requiring us to know the full spectrum in advance.

### One dissipative reset cycle

During the interaction window we evolve under

$$
H(t)=H_S+H_E+\alpha f(t)H_{SE}.
$$

For the hardware-oriented implementation used throughout this notebook,

$$
\alpha=1.75,\qquad T=0.75,\qquad N_{\rm Trot}=3,
$$

and $f(t)$ is effectively constant ($\sigma\rightarrow\infty$). These deliberately shallow parameters make the channel practical on present hardware, but they also introduce a **fixed-point error**: even without hardware noise, the steady state need not equal the exact Gibbs state.

### Mixing time versus fixed-point accuracy

Two separate questions will appear repeatedly:

- **How fast do we approach the steady state?** This is controlled by the mixing time $\tau_{\rm mix}$.
- **How close is that steady state to the desired Gibbs state?** This is the fixed-point accuracy.

A channel can converge quickly to an imperfect thermal state, or converge slowly to a more accurate one. The hardware-oriented parameters above intentionally trade some ideal fixed-point accuracy for stronger, shallower dissipation.

This distinction is especially important later:

$$
\boxed{\text{exact Gibbs target}}
\;\neq\;
\boxed{\text{noiseless dissipative steady state}}
\;\neq\;
\boxed{\text{noisy QPU steady state}}
$$

in general.

### Dissipative Quantum Gibbs Sampling Workflow

<div class="alert alert-block alert-info">

**Step 1: Prepare the <span style="color:darkgreen">System</span> in a random product state.**  
This corresponds to a state sampled from the Gibbs ensemble for $\beta = 0$.

**Step 2: Apply a quantum channel $\Phi$ that couples the <span style="color:darkgreen">System</span> to the <span style="color:red">Environment</span> a total of $N_{\text{resets}}$ times.**  
Each application of $\Phi$ involves three steps:  
(a) Prepare the <span style="color:red">**Environment**</span> in a product state sampled from the Gibbs ensemble of the <span style="color:red">Environment</span> Hamiltonian $H_E$.  
(b) Evolve the <span style="color:darkgreen">**System**</span> + <span style="color:red">**Environment**</span> under the total Hamiltonian for a time $T$.  
(c) Measure the <span style="color:red">**Environment**</span> qubits and reset them to $|0\rangle$.

**Step 3: Measure the energy of the <span style="color:darkgreen">System</span> and compute the steady state energy density.**  
That is, estimate: $E_{\text{s.s.}} = \langle \psi_{\text{s.s.}}| H_{\text{AFIM}} | \psi_{\text{s.s.}} \rangle$ and compute $E_{\text{s.s.}} / N_S$.

***NOTE that this algorithm is stochastic; we will rely on repeating the random process many times (and averaging the resulting expectation values) to reach convergence.***

</div>

<p align="center">
    <img src="./figures/algorithm.png" width="1000">
</p>

### Your Task

<div class="alert alert-block alert-success">

**Use the dissipative quantum Gibbs sampling workflow to prepare thermal states at specific values of $\beta$ and estimate the resulting energy density for the antiferromagnetic Ising model (AFIM) on a kagome lattice.**

**Part 1 (Statevector Simulation):**
- $(L_x, L_y) = (2, 2) \longrightarrow N_S = 12$; $N_E = \{1, 3\}$ (a total of 13, 15 qubits)
- $N_{\text{resets}} = \{0, 1, 2, ..., 50\}$
- $\beta = \{1/4, 100\}$

**Part 2 (Small-scale Hardware Experiment, without Error Mitigation):**
- $(L_x, L_y) = (2, 2) \longrightarrow N_S = 12$; $N_E = 12$ (a total of 24 qubits)
- $N_{\text{resets}} = \{0, 1, 3, 7\}$
- $\beta = \{1/4, 100\}$

**Part 3 (Small-scale Hardware Experiment, with Error Mitigation):**
- $(L_x, L_y) = (2, 2) \longrightarrow N_S = 12$; $N_E = 12$ (a total of 24 qubits)
- $N_{\text{resets}} = 7$
- $\beta = \{1/4, 100\}$

**Part 4 (Large-scale Hardware Experiment, with Error Mitigation):**
- $(L_x, L_y) = (3, 3) \longrightarrow N_S = 36$; $N_E = 31$ (a total of 67 qubits)
- $N_{\text{resets}} = 7$
- $\beta = 100$
</div> 

### Grading Criteria

<div class="alert alert-block alert-warning">

The bulk of the challenge will be graded based on values for the **energy density** $E/N_S$.

**10 Points: $E/N_S$ for statevector simulations of $(2,2)$ kagome lattice (Part 1)**
- 2.5 points for $E/N_S$ results for each $(\beta, N_E)$ (*Part 1.2*)

**40 Points: $E/N_S$ for QPU simulation of $(2,2)$ kagome lattice (Part 2)**
- 3.5 points for correct <span style="color:red">**Environment**</span>  initialization (*Part 2.6*)
- 1.5 points for correct <span style="color:darkgreen">**System**</span> initialization, `init_system` (*Part 2.6*)
- 2 points for correct <span style="color:darkgreen">**System**</span>-<span style="color:red">**Environment**</span> couplings (*Part 2.7*)
- 3.5 points for correct `evolve_HSE` function (*Part 2.7*)
- 2 points for correct `evolve_HX` function (*Part 2.7*)
- 2 points for correct `evolve_HZ` function (*Part 2.7*)
- 2 points for correct `one_Trot_step` function (*Part 2.7*)
- 1.5 points for correct `compile_to_isa` function (*Part 2.8A*)
- 1.5 points for correct `box_and_build` function (*Part 2.8B*)
- 1.5 points for correct `samplex_parameter_values` function (*Part 2.8C*)
- 1.5 points for correct `finalize_templates` function (*Part 2.8E*)
- 1 point for the two-qubit circuit depth (*Part 2.8F*)
- 1.5 points for local Samplex sampling and the training `QuantumProgram` (*Part 2.8D and 2.8G*)
- The conceptual questions in Part 2.8 give feedback only (0 points)
- 2.5 points for correct `get_vals` function (*Part 2.11*)
- 2.5 points for correct `compute_observables` function (*Part 2.11*)
- 5 points for $E/N_S$ with cold bath, based on best performance (comparison of submitted "best performance" file to theoretical simulation)  (*Part 2.14*)
- 5 points for $E/N_S$ with warm bath, based on best performance (comparison of submitted "best performance" file to theoretical simulation)  (*Part 2.14*)

**20 Points: Error mitigated $E/N_S$ for QPU simulation of $(2,2)$ kagome lattice (Part 3)**
- 5 points for correct `linear_zne` function (*Part 3.4*)
- 5 points for correct `perform_zne` function (*Part 3.4*)
- 5 points for $E/N_S$ with cold bath, based on best performance (comparison of submitted "best performance" file to theoretical simulation)  (*Part 3.5*)
- 5 points for $E/N_S$ with warm bath, based on best performance (comparison of submitted "best performance" file to theoretical simulation)  (*Part 3.6*)

**30 Points: Error mitigated $E/N_S$ for QPU simulation of $(3,3)$ kagome lattice (Part 4)**
- 5 points for correct large-scale addition to `kagome_edge_set` (*Part 4.1*)
- 5 points for correct large-scale addition to `SE_Coupling` (*Part 4.1*)
- 5 points for the large-scale two-qubit circuit depth (*Part 4.2*)
- 15 points for $E/N_S$ with cold bath, based on best performance (comparison of submitted "best performance" file to theoretical simulation)  (*Part 4.7*)

*The exercises in Parts 2.8D and 2.8G check local Samplex sampling and construction of a training `QuantumProgram`; they do not require submitting a QPU job. The QPU-derived energy results in Parts 2.14, 3.5, 3.6, and 4.7 are submitted from your own experiments. Each graded exercise keeps your best score, so re-running its grading cell with a worse result does not lower your existing score.*

**Winning Team**

The winning team with the highest total score will have the opportunity to present their solution notebook to the other contestants. 

***Good luck and have fun!***

</div> 

# **PART 1: Statevector Simulation**

In this part of the challenge, you will use the provided code to compute the steady state energy of the AFIM on a small kagome lattice $(L_x, L_y) = (2, 2)$ ($N_S = 12$). As is done in the original paper, we will use $g_x = 0.5$, $g_z = 2.0$, placing the <span style="color:darkgreen">System</span> in the frustrated ice phase in the low-temperature regime. 

### 1.1 Build the kagome AFIM

To familiarize yourself with the algorithm, we have provided code in `statevector_thermal_prep.py` that implements a statevector simulation of the workflow (this is a simplified version of the code provided by the authors in [this repo](https://github.com/Roland-Farrell/dissipative_Gibbs_sampling)). All four runs below use the same <span style="color:darkgreen">system</span> Hamiltonian; only the bath (<span style="color:red">Environment</span>) temperature and bath size ($N_E$) change.

In [ ]:
from utils.statevector_thermal_prep import kagome_ising, run_thermal_prep
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Folders for saved results and QPU job IDs
for folder in ("submitted_results", "job_info"):
    Path(folder).mkdir(exist_ok=True)

Lx = 2
Ly = 2
gx = 0.5
gz = 2.0

model = kagome_ising(Lx=Lx, Ly=Ly, gx=gx, gz=gz, pbc=False) ### PBC = False for open boundary conditions
print(f"System qubits: NS = {model.NS}")

### 1.2 Run the dissipative dynamics at two temperatures

We compare a high-temperature bath, $\beta=0.25$, with a low-temperature bath, $\beta=100$ (used as a numerical proxy for $\beta=\infty$). We also compare an <span style="color:red">Environment</span> comprised of one and three qubits.

The primary function to implement a thermal state prep simulation is `run_thermal_prep`, which takes the following inputs: 
- `model`: the model of the <span style="color:darkgreen">System</span> (we'll use the kagome AFIM prepared in the previous cell)
- `alpha`: the <span style="color:darkgreen">System</span>-<span style="color:red">Environment</span> coupling (1.75)
- `beta`: the inverse temperature (100 to approximate $\beta = \infty$)
- `t_tot`: the total evolution time $T$ (0.75)
- `nTrot`: the number of Trotter steps (3)
- `nSteps`: the number of applications of $\Phi$ (use 50 for Part 1)
- `nSamples`: the number of randomizations of the algorithm (use 100 for Part 1)
- `NE`: the number of <span style="color:red">Environment</span> qubits $N_E$
- `omega_max`: the upper bound on the randomly sampled Bohr frequencies (use $\max(4g_x, 4g_z, 4)$)

The function returns the total energy of the <span style="color:darkgreen">System</span> for each application of $\Phi$ (axis 0) for each randomization (axis 1).

<div class="alert alert-block alert-success">

✏️ **Your task.** In the cell below, fill in the missing code to implement the algorithm on a small instance of the kagome lattice for two different values of `NE`, which corresponds to the number of <span style="color:red">Environment</span> qubits $N_E$.

</div>

*Note that the following cell will take approximately 3 minutes to run.*

In [ ]:
import pickle
from pathlib import Path

alpha = ...  # TODO: System-Environment coupling (see the parameter list above)
t_tot = ...  # TODO: total evolution time T
nTrot = 3
nSteps = ...  # TODO: number of applications of Phi for Part 1
nSamples = ...  # TODO: number of randomizations for Part 1
omega_max = ...  # TODO: upper bound on the Bohr frequencies, in terms of gx and gz
PART1_BETAS = (0.25, 100.0)
PART1_NES = (1, 3)

path = Path('./submitted_results/part1_energies.pkl')
if path.exists():
    with open(path, 'rb') as file:
        part1_energies = pickle.load(file)
else: 
    part1_energies = {}
    for beta in PART1_BETAS:
        for NE in PART1_NES:
            print(f"\nRunning beta={beta}, NE={NE}")
            part1_energies[(beta, NE)] = run_thermal_prep(
                model=model,
                alpha=alpha,
                beta=beta,
                t_tot=t_tot,
                nTrot=nTrot,
                nSteps=nSteps,
                nSamples=nSamples,
                NE=NE,
                omega_max=omega_max,
                seed=1000 + int(100 * beta) + NE,
                progress=False,
            )/model.NS

Run the cell below to see how the energy density approaches steady state with an increasing number of `nSteps` $= N_{\text{resets}}$. Here, we take the mean over `axis=1` to average over randomizations.  

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True, dpi=400)
resets = np.arange(nSteps + 1)

for ax, beta in zip(axes, PART1_BETAS):
    for NE in PART1_NES:
        E = part1_energies[(beta, NE)]
        mean = E.mean(axis=1)
        sem = E.std(axis=1, ddof=1) / np.sqrt(E.shape[1])
        ax.plot(resets, mean, label=fr"$N_E={NE}$")
        ax.fill_between(resets, mean - sem, mean + sem, alpha=0.2)
    ax.set_title(fr"$\beta={beta}$")
    ax.set_xlabel(r"$N_{\rm resets}$")
    ax.grid(alpha=0.3)
    ax.legend()

axes[0].set_ylabel(r"$E/N_S$")
fig.suptitle("Dissipative cooling of the 12-spin kagome AFIM")
fig.tight_layout();

The two comparisons isolate different physics:

- Changing $\beta$ changes **where** the dissipative dynamics tends to settle: a colder bath produces a lower-energy steady state.
- Increasing $N_E$ changes primarily **how quickly** the steady state is approached.

Notice that by increasing the size of the coupled <span style="color:red">Environment</span>, steady state is reached with fewer steps required. In the weak-coupling regime, the convergence curves approximately collapse when the reset count is rescaled by $N_E/N_S$. 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True, dpi=400)

for ax, beta in zip(axes, PART1_BETAS):
    for NE in PART1_NES:
        E = part1_energies[(beta, NE)]
        x = resets * NE / model.NS
        ax.plot(x, E.mean(axis=1), label=fr"$N_E={NE}$")
    ax.set_title(fr"$\beta={beta}$")
    ax.set_xlabel(r"$N_{\rm resets}N_E/N_S$")
    ax.grid(alpha=0.3)
    ax.legend()

axes[0].set_ylabel(r"$E/N_S$")
fig.suptitle("Rescaling the reset count by environment density")
fig.tight_layout();

The curves above demonstrate dissipative cooling and the role of <span style="color:red">Environment</span> size. They do **not** by themselves prove that the fixed point is the exact Gibbs state. With $\alpha=1.75$, $T=0.75$, and only three Trotter steps per reset cycle, the noiseless channel has a noticeable systematic error. That is intentional: the same shallow parameters are used later because they are much more favorable for hardware.

From this point onward, we will therefore use the phrase **noiseless dissipative reference** when we mean the output of this approximate channel, and reserve **exact Gibbs state** for $e^{-\beta H}/Z$.

<div class="alert alert-block alert-info">

**Save your energy density results (2.5 pts per $(\beta, N_E)$) using the cell below, and then check your score.**

</div>

In [ ]:
import pickle
with open('submitted_results/part1_energies.pkl', "wb") as f: 
    pickle.dump(part1_energies, f)

In [ ]:
grade_lab_thermal_states_ex1_2(part1_energies)

## *Bridge — What can we measure on a QPU?*

A useful feature of the AFIM is that its energy can be reconstructed from only **two measurement bases**:

$$
H_S=\sum_{\langle i,j\rangle}Z_iZ_j+g_x\sum_i X_i+g_z\sum_i Z_i.
$$

A $Z$-basis measurement gives both

$$\langle Z_i\rangle\quad\text{and}\quad\langle Z_iZ_j\rangle,$$

while an $X$-basis measurement gives

$$\langle X_i\rangle.$$

Therefore

$$
\frac{E}{N_S}=\frac{1}{N_S}\left[
\sum_{\langle i,j\rangle}\langle Z_iZ_j\rangle
+g_x\sum_i\langle X_i\rangle
+g_z\sum_i\langle Z_i\rangle
\right].
$$

The same data also gives the magnetizations

$$M_Z=\frac{1}{N_S}\sum_i\langle Z_i\rangle,\qquad
M_X=\frac{1}{N_S}\sum_i\langle X_i\rangle,$$

and connected nearest-neighbor correlations. This will let us diagnose thermal-like behavior without reconstructing the full density matrix.

# **PART 2: Small-Scale Hardware Experiment**

We now turn to a quantum hardware experiment. We will study the same small <span style="color:darkgreen">System</span> introduced in Part 1, but substantially enlarge the <span style="color:red">Environment</span> to accelerate convergence. This reduces the required two-qubit gate depth and, consequently, improves the experimental signal.

Specifically, we keep the same $N_S=12$ kagome <span style="color:darkgreen">System</span>, while increasing the <span style="color:red">Environment</span> to $N_E=12$. The full experiment therefore uses **24 active qubits**.

### 2.1 The Lattice
The kagome lattice can easily be embedded on the heavy-hex coupling map of a Heron QPU by placing the <span style="color:darkgreen">System</span> qubits between the "heavy" nodes that have degree-3 connectivity. The <span style="color:red">Environment</span> qubits can be embedded in between these nodes and along the edge of the lattice. The <span style="color:red">Environment</span> qubits are paired with up to two neighboring <span style="color:darkgreen">System</span> qubits (this is only necessary for larger lattices). 

Examples for the two target <span style="color:darkgreen">Systems</span> (Left: Parts 2 and 3, Right: Part 4) are provided below. Here, <span style="color:darkgreen">System</span> qubits are shown in dark green and <span style="color:red">Environment</span> qubits are shown in red. The edges of the kagome lattice are indicated by the dark green lines (sometimes crossing <span style="color:red">Environment</span> qubits), and the <span style="color:darkgreen">System</span>-<span style="color:red">Environment</span> pairs are indicated by the red chain-links. (In the larger lattice, some chain-links involve the same <span style="color:red">Environment</span> qubit -- only one <span style="color:darkgreen">System</span>-<span style="color:red">Environment</span> interaction is implemented with each application of $\Phi$, where the <span style="color:darkgreen">System</span> qubit involved is selected randomly). The blue lines represent the native hardware connectivity for heavy-hex.

<p align="center">
    <img src="./figures/small-scale_virtual-layout.png" width="400"> &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; <img src="./figures/large-scale_virtual-layout.png" width="550">
</p>


#### *A note on qubit enumeration* 

In the figures above, the virtual qubit layouts are shown. We enumerate the <span style="color:darkgreen">System</span> qubits as they appear in the lattice from left to right, top to bottom, starting from index 0. We then enumerate the <span style="color:red">Environment</span> qubits starting at index $N_S$ based on the <span style="color:darkgreen">System</span> qubit pairing shown by the red chain links (that is, the <span style="color:red">Environment</span> qubit $N_S$ is the one that couples with <span style="color:darkgreen">System</span> qubit 0, the <span style="color:red">Environment</span> qubit $N_S + 1$ is the one that couples with <span style="color:darkgreen">System</span> qubit 1, and so on). This convention will be used when building circuits and enumerating interactions below. 

However, when we embed the lattice onto a QPU, we need to map this enumeration to the qubit enumeration used for the device. This is the *physical layout*. We have some freedom to translate the lattice around the hardware coupling map; you should pick a physical layout to avoid qubits and edges that have high error rates. A physical layout for the small-scale <span style="color:darkgreen">System</span> is provided below, where <span style="color:darkgreen">System</span> qubit 0 of the virtual layout has been mapped to the physical qubit 38, <span style="color:red">Environment</span> qubit 12 of the virtual layout has been mapped to the physical qubit 29, and so on.

<p align="center">
    <img src="./figures/small-scale_hardware-layout.png" width="600">
</p>

### 2.2 Load the precomputed noiseless $12+12$ references

The goal of Part 2 is **controlled validation**. We have precomputed noiseless statevector trajectories for the **same $12+12$ dissipative channel**, allowing us to disentangle hardware-induced errors from the channel's intrinsic fixed-point error.

For both $\beta=0.25$ and $\beta=100$, the reference datasets contain reset depths

$$N_{\rm resets}=0,1,\ldots,7.$$

Each combined file is formed from 1000 independent statevector trajectories total. These are **noiseless dissipative-channel references**. In case you played with the algorithm / system parameters in Part 1, we'll redefine them below.

In [ ]:
from pathlib import Path

alpha = 1.75
t_tot = 0.75
nTrot = 3

gx = 0.5
gz = 2.0
Lx, Ly = 2, 2
NS = 12
SMALL_BETAS = (0.25, 100.0)
BASELINE_FILES = {
    0.25: Path("./statevector_data/statevector_12_12_b0_25.npy"),
    100.0: Path("./statevector_data/statevector_12_12_b100.npy"),
}

statevector_reference = {
    beta: np.load(path)
    for beta, path in BASELINE_FILES.items()
}

for beta, arr in statevector_reference.items():
    print(f"beta={beta:6g}: shape={arr.shape}, trajectories={arr.shape[1]}")

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.3), sharey=True, dpi=400)
sv_resets = np.arange(8)

for ax, beta in zip(axes, SMALL_BETAS):
    arr = statevector_reference[beta] / NS
    mean = arr.mean(axis=1)
    sem = arr.std(axis=1, ddof=1) / np.sqrt(arr.shape[1])
    ax.plot(sv_resets, mean, marker="o", label="noiseless dissipative reference")
    ax.fill_between(sv_resets, mean - sem, mean + sem, alpha=0.2)
    ax.set_title(fr"$\beta={beta}$")
    ax.set_xlabel(r"$N_{\rm resets}$")
    ax.grid(alpha=0.3)

axes[0].set_ylabel(r"$E/N_S$")
axes[1].legend()
fig.tight_layout();

### 2.3 The hardware pipeline at a glance

In Part 1, a statevector simulator let us specify the dissipative channel $\Phi$ directly. A real quantum processor does not accept the instruction “apply $\Phi$”. It accepts a **timed sequence of physical gates, measurements and resets** on particular hardware qubits. Our job is to turn the Gibbs-sampling protocol into that sequence, then gather enough noisy measurement samples to estimate an observable.

As we translate the Gibbs-sampling protocol into hardware circuits, we encounter several sources of randomness, each of them playing a different role in the experiment that follows: 

1. **Algorithmic randomness:** the initial <span style="color:darkgreen">System</span> bitstring, the <span style="color:red">Environment</span> frequencies and thermal occupations, and the randomly chosen jump operators. This randomness **defines one realization of the physical Gibbs-sampling experiment**; averaging over realizations is part of the intended thermalization protocol.
2. **Pauli-twirling randomness:** randomized, compensated Pauli operations around suitable gate layers and measurements. These give *different implementations of the same intended logical operation*, helping turn coherent error into more stochastic-looking error. Twirling does not, by itself, prepare a thermal state.
3. **Shot noise:** even after the circuit and its randomizations are fixed, repeated measurements give different bitstrings. We take many shots to estimate expectation values.

The notebook builds the hardware workload through the following stages:

| Stage | What happens here | Where to see it |
|---|---|---|
| **1. Prepare a logical circuit** | Build random initial <span style="color:darkgreen">System</span> states, thermally sample the <span style="color:red">Environment</span>, and implement the Trotterized $H_S+H_E+\alpha f(t)H_{SE}$ evolution and bath reset windows. | Parts 2.6–2.7; `build_logical_circuits(...)` |
| **2. Choose physical qubits and transpile** | Map virtual qubits to a specific Heron heavy-hex layout and translate/reroute operations into the selected backend's instruction set architecture (**ISA**). | Parts 2.5 and 2.8; `compile_to_isa(...)` |
| **3. Box the ISA circuit** | Group gate layers and measurements into annotated *boxes* describing where randomized compilation should act. | Part 2.8; `generate_boxing_pass_manager(...)` |
| **4. Build a template and a Samplex** | Samplomatic turns the boxed circuit into a parameterized **template circuit** plus a **Samplex**, which specifies how to sample twirls and how to correct randomized measurements. | Part 2.8; `samplomatic.build(...)` |
| **5. Finish the physical schedule** | Add the `meas_ps` leakage-check register via `xslow`, restore barriers around resets, use backend gate durations to schedule operations, and insert dynamical-decoupling (DD) $X$ pulses in suitable idle windows. | Part 2.8; `finalize_templates(...)` |
| **6. Assemble the hardware workload** | Combine stages 1-5 to produce the scheduled circuit templates, corresponding samplexes, and measurement-basis settings. | Part 2.8; `generate_gibbs_circuit(...)` |
| **7. Execute and analyze** | Package the template/Samplex pairs as `QuantumProgram` items, send them using `Executor`, retrieve arrays of measured bits and correction bits, and evaluate $\langle Z_i\rangle$, $\langle X_i\rangle$, $\langle Z_iZ_j\rangle$. | Parts 2.9–2.14 |

Further reading:
- [IBM Quantum — transpile with pass managers](https://quantum.cloud.ibm.com/docs/en/guides/transpile-with-pass-managers)
- [Samplomatic — boxing/transpiler](https://quantum.cloud.ibm.com/docs/en/addons/samplomatic/guides/transpiler)
- [Samplex inputs and outputs](https://quantum.cloud.ibm.com/docs/en/addons/samplomatic/guides/samplex-io)
- [Executor quickstart](https://quantum.cloud.ibm.com/docs/en/guides/get-started-with-executor)

<div class="alert alert-block alert-success">

✏️ **Your task.** In the next cell, set `BACKEND_NAME` to the IBM Heron r3 backend you will use, then inspect `CONNECT_TO_QPU`, `RUN_QPU_SMALL`, `N_RANDOMIZATIONS`, `N_INIT`, `N_TWIRLS` and `N_SHOTS`. **Connecting and submitting are different actions.** `CONNECT_TO_QPU` permits backend-aware compilation and retrieval; only the submission switch `RUN_QPU_SMALL` enables new QPU jobs. Increasing the sweep dimensions increases the number of circuits/shots and therefore the hardware cost.

</div>

<div class="alert alert-block alert-warning">

💡 **Hint.** We recommend `ibm_boston`: the hardware layout in Part 2.5 was chosen for it. If you use a different backend, adjust the layout in Part 2.5 to avoid qubits and edges with high error rates.

</div>

In [ ]:
# --- Two safety switches -----------------------------------------------------
# Submitting jobs to a real QPU costs time and money, so RUN_QPU_SMALL is OFF by default.
#   CONNECT_TO_QPU : open a connection. Needed to build circuits, inspect them,
#                    and retrieve already-finished jobs.
#   RUN_QPU_SMALL  : actually submit NEW jobs.
CONNECT_TO_QPU = True
RUN_QPU_SMALL = False

# --- Backend / experiment settings ------------------------------------------
BACKEND_NAME   = ...  # TODO: the IBM Heron r3 backend (see the task above)

N_RANDOMIZATIONS = 8   # independent circuit randomizations (fresh instantiation), averaged over
N_INIT           = 5   # random beta=0 initial bitstrings per job
N_TWIRLS         = 10  # Pauli twirls of each initialization, bundled into one job
N_SHOTS          = 64  # measurement shots per (init, twirl)
BASE_SEED        = 42  # reproducibility

# Cooling points for the two bath temperatures.
SMALL_RESETS = (0, 1, 3, 7)

print("CONNECT_TO_QPU =", CONNECT_TO_QPU, "  RUN_QPU =", RUN_QPU_SMALL)

### 2.4 Connect to the backend

Before executing our circuits, we need to connect to an IBM Quantum QPU and verify that it supports the required gates. Qiskit's `backend.target` describes the instructions available on the selected hardware.

Three instructions are particularly important for this challenge:

- **`xslow` — Extended-duration X gate:** Performs a bit flip like a standard X gate, but over a longer duration. We use it for **non-Markovian error checks**, which can help identify shots affected by certain hardware errors. In Part 2.8, these checks generate additional measurements (`meas_ps`), which we use for **post-selection in Part 3**.  

- **`rx` — Single-qubit X rotation:** Implements a continuously parameterized rotation,

  $$
  R_X(\theta)=e^{-i\theta X/2}.
  $$

  These rotations implement the single-qubit terms in our Trotterized Hamiltonian evolution, the random initial System bitstrings, and the final readout-basis rotation.

- **`rzz` — Two-qubit ZZ rotation:** Implements an interaction between two qubits,

  $$
  R_{ZZ}(\theta)=e^{-i\theta Z\otimes Z/2}.
  $$

  These gates implement the System $ZZ$ bonds and, sandwiched between single-qubit basis changes, the System–Environment couplings.

**Why `use_fractional_gates=True`?**

This option exposes supported fractional `rx` and `rzz` rotations directly to Qiskit's transpiler. On compatible IBM hardware, this can reduce circuit depth and gate overhead compared with decomposing these operations into fixed-angle gates.

The following code checks whether all three instructions are available on our selected backend. All three checks must return `True`: without `xslow`, `finalize_templates` raises an error. 

**Further reading:**
- [IBM Quantum: Fractional gates](https://quantum.cloud.ibm.com/docs/en/guides/fractional-gates)
- [IBM Quantum: Postselection with non-Markovian error checks](https://quantum.cloud.ibm.com/docs/en/addons/qiskit-mitigation/guides/postselection-with-non-markovian-error-checks)

In [ ]:
service = None
backend = None

if CONNECT_TO_QPU:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()
    backend = service.backend(
        name=BACKEND_NAME,
        use_fractional_gates=True     # exposes the fractional rx / rzz gates the circuit uses
    )
    print("Connected to:", backend.name)
    print("  'xslow' in target      :", "xslow" in backend.target)     # needed for post-selection
    print("  'rx'  in basis_gates   :", "rx"  in backend.basis_gates)
    print("  'rzz' in basis_gates   :", "rzz" in backend.basis_gates)
else:
    print("CONNECT_TO_QPU is False -> not connecting.")
    print("Set it to True (with an IBM account configured) to build circuits or retrieve jobs.")

### 2.5 Qubit layout

The kagome <span style="color:darkgreen">System</span> is embedded on the device's heavy-hex grid by the hard-coded map in
`hardware_layouts(Lx, Ly)` (illustrated in the Part 2.1 figures). The returned layout obeys the following convention:

- the **first `NS` entries** are the <span style="color:darkgreen">System</span> qubits (`0..NS-1`),
- the **remaining entries** are the <span style="color:red">Environment</span> qubits.

These qubits are ordered the same way as in the virtual layout, but now, the index refers to the qubit index on the device.

`kagome_edge_set(Lx, Ly)` returns the list of nearest-neighbor `(i, j)` **bonds** of the kagome lattice (that is, exactly the $\langle i,j\rangle$ pairs in the $\sum Z_iZ_j$ term of $H_S$). This cell is pure bookkeeping and needs no backend.

<div class="alert alert-block alert-success">

✏️ **Your task.** Check the hard-coded layout in `hardware_layouts` against the most recent calibration of your backend (the cell after it lists the highest-error CZ edges among the selected qubits). If needed, adjust the layout to avoid qubits and edges with high error rates, keeping the System-then-Environment ordering.

</div>

In [ ]:
def hardware_layouts(Lx,Ly):
    # convention: [system qubits, left to right, top to bottom] + [environment qubits in the order that they are paired with the ordered system qubits]
    layout_dict = {(2,2):[38,39,48,50,52,54,57,58,66,68,70,72] + [29,33,49,51,53,55,47,71,65,67,69,73],
                   }
    return layout_dict[(Lx,Ly)]

def kagome_edge_set(Lx,Ly):
    # returns a list of the System coupling edges <i,j> for a particular kagome lattice of size (Lx, Ly).
    edge_set_dict = {(2,2):[(0,2),(0,3),(1,4),(1,5),(2,3),(3,4),(4,5),(2,6),(3,7),(4,7),(6,8),(6,9),(7,10),(7,11),(8,9),(9,10),(10,11)],
                    }
    return edge_set_dict[(Lx,Ly)]

NS = 12                                  # 12 system qubits
layout = hardware_layouts(Lx, Ly)        # physical qubits: system first, then bath
system_qubits = layout[:NS]
env_qubits = layout[NS:]
edge_indices = kagome_edge_set(Lx, Ly)   # the <i,j> ZZ bonds of the kagome lattice

NE  = len(env_qubits)
Nbonds = len(edge_indices)
nbits  = NS + NE                # total measured qubits (system + bath)

print(f"system qubits     : {NS:2d}  -> {system_qubits}")
print(f"environment qubits: {NE:2d}  -> {env_qubits}")
print(f"ZZ bonds          : {Nbonds}")
print(f"measured qubits (nbits) : {nbits}")

You can also check the errors on the selected qubits on the backend you specify. 

In [ ]:
if backend is not None:
    target = backend.target
    active = set(layout)
    cz_data = []
    if "cz" in target:
        for pair, properties in target["cz"].items():
            if pair is not None and len(pair) == 2 and set(pair) <= active:
                error = None if properties is None else properties.error
                if error is not None:
                    cz_data.append((pair, float(error)))
    cz_data.sort(key=lambda entry: entry[1], reverse=True)
    print("Highest-error CZ edges among selected qubits:")
    for pair, error in cz_data[:5]:
        print(f"  {pair}: error={error:.5f}")
else:
    print("Connect to the QPU backend to inspect its Target; no jobs are submitted.")

### 2.6 The dissipative quantum Gibbs sampling workflow: State Preparation

We now write the **initialization building blocks** that the circuit generator will call. There are two distinct thermal distributions here: the **initial <span style="color:darkgreen">System</span>** is sampled at $\beta=0$ (infinite temperature), whereas the **<span style="color:red">Environment</span>** is sampled at the bath inverse temperature $\beta$ selected for the experiment. The bath temperature is what biases the subsequent dissipative dynamics toward cold or warm states.

**1. Initialize the <span style="color:darkgreen">System</span>.**

We begin by preparing the <span style="color:darkgreen">System</span> in an infinite-temperature state ($\beta=0$), corresponding to the maximally mixed density matrix. For $N_S$ qubits, this can be expressed as an equal mixture of all computational-basis states:
$$
\rho_{S,0}=\frac{I}{2^{N_S}}=\frac{1}{2^{N_S}}\sum_{b\in\{0,1\}^{N_S}}|b\rangle\langle b|.
$$

Rather than preparing this mixed state directly, we can reproduce it statistically by randomly sampling computational-basis states and averaging over the resulting realizations. To implement this efficiently, we use parameterized $R_X(\theta)=e^{-i\theta X/2}$ rotations, where $\theta=0$ prepares $|0\rangle$ and $\theta=\pi$ prepares $|1\rangle$, up to a global phase. The function `init_system(circ, NS)` therefore assigns a symbolic rotation parameter to each System qubit using `ParameterVector("init", NS)`. These parameters are later sampled independently from $\{0,\pi\}$ through `samplex_parameter_values(...)` in Part 2.8, allowing us to generate different initial bitstrings while maintaining consistent initialization across twirls and measurement bases.

<div class="alert alert-block alert-success">

✏️ **Your task.** In the cell below, complete `init_system(circ, NS)` so that it prepares a random infinite-temperature ($\beta=0$) computational-basis state with a **single parameterized circuit**: one $R_X$ rotation per System qubit, with angle `init[n]`.

</div>

In [ ]:
from qiskit.circuit import ParameterVector, QuantumCircuit

def init_system(circ, NS):
    init = ParameterVector("init", NS)
    for n in range(NS):
        ...  # TODO: apply RX(init[n]) to system qubit n
    return circ

_demo_init = init_system(QuantumCircuit(2), 2)
assert {p.name for p in _demo_init.parameters} == {"init[0]", "init[1]"}
print("Parameterized initialization:", _demo_init.count_ops())


**2. Initialize the <span style="color:red">Environment</span>.**

Unlike the System, which begins at infinite temperature, the <span style="color:red">Environment</span> must be initialized according to the bath temperature $\beta$ specified for the experiment. As introduced in the background, the bath Hamiltonian is

$$
H_E=-\sum_{n=0}^{N_E-1}\frac{\omega_n}{2}Z_n.
$$

Each Environment qubit therefore has two possible energies, $-\omega_n/2$ and $+\omega_n/2$, corresponding to the computational states $|0\rangle$ and $|1\rangle$. At inverse temperature $\beta$, the probability of occupying $|1\rangle$ follows the Boltzmann distribution:

$$
P(b_n=1)=\frac{e^{-\beta\omega_n/2}}{e^{\beta\omega_n/2}+e^{-\beta\omega_n/2}}
=\frac{1}{1+e^{\beta\omega_n}}.
$$

We use `sample_boltzmann_zn(omega_n, beta)` to sample each Environment bit according to this probability. The function `init_environment(circ, NS, omega_n, beta)` then prepares the corresponding computational-basis state by applying an $X$ gate to Environment qubit `NS+n` whenever the sampled bit is 1. Repeating this procedure produces an ensemble of independently sampled thermal product states, consistent with the bath's thermal density matrix. In particular, for positive $\omega_n$, the Environment increasingly favors $|0\rangle$ as $\beta$ increases, whereas at $\beta=0$ both computational states are equally likely.

<div class="alert alert-block alert-success">

✏️ **Your task.** In the cell below, fill in the functions `sample_boltzmann_zn` and `init_environment` to perform the <span style="color:red">Environment</span> initialization.

</div>

In [ ]:
from qiskit.circuit import QuantumCircuit

# Sample computational basis states (as a bitstring) from e^{-beta H_E}.
# Bits in the bitstring are ordered as q_{N-1} q_{N-2} ... q_1 q_0 (little endian notation).
def sample_boltzmann_zn(omega_n, # randomly sampled Bohr frequencies
                        beta,    # inverse temperature
                        ):
    '''
    TODO: for each Environment qubit n, sample bit n = 1 with probability 1/(1+e^{beta*omega_n[n]})
          (and 0 otherwise). Return the bits as a string ordered q_{N-1} ... q_1 q_0.
    '''
    return ...

# Initialize the Environment qubits in a randomly sampled state from the Boltzmann distribution for H_E.
# circ.<gate> appends gates to circ in place; this function returns nothing.
def init_environment(circ,    # QuantumCircuit object
                     NS,      # number of system qubits
                     omega_n, # randomly sampled Bohr frequencies
                     beta,    # inverse temperature
                     ):
    '''
    TODO: sample the Environment bits with sample_boltzmann_zn, then apply X to qubit NS+n
          wherever bit n is 1 (remember the string is ordered q_{N-1} ... q_0).
    '''
    ...

<div class="alert alert-block alert-info">

**Score your implementation below (5 pts total: 3.5 for `init_environment`, 1.5 for `init_system`).**

</div>

In [ ]:
grade_lab_thermal_states_ex2_6(init_environment, init_system)

### 2.7 The dissipative quantum Gibbs sampling workflow: Trotterization

Next, we need to implement the Trotterization of the collective <span style="color:darkgreen">System</span> and <span style="color:red">Environment</span>. We are going to break this down into small pieces, and then put everything together into one function `H_tot_trot`. 

Recall the total Hamiltonian: 

$$
\begin{aligned}
H(t) &= H_S+H_E+\alpha f(t)H_{SE} \\ 
&= \sum_{\langle i_S, j_S \rangle} Z_{i_S} Z_{j_S} + \sum_{i_S} (g_x X_{i_S} + g_z Z_{i_S}) 
\\ &\qquad \qquad-\sum_{i_E}\frac{\omega_{i_E}}{2}Z_{i_E} 
\\ &\qquad \qquad+ \alpha f(t) \sum_{\langle i_S,i_E\rangle} O_{i_S}\otimes X_{i_E}, \quad \text{where}\quad O_{i_S}\in\{\pm X_{i_S},\pm Y_{i_S},\pm Z_{i_S}\}.
\end{aligned}
$$

To Trotterize this, we can rearrange the terms into groups of mutually commuting terms: 
$$
H(t) = H_Z+H_X+\alpha f(t)H_{SE} , 
$$

where: 
$$
\begin{aligned}
H_Z &= \sum_{\langle i_S, j_S \rangle} Z_{i_S} Z_{j_S} + \sum_{i_S}  g_z Z_{i_S} - \sum_{i_E}\frac{\omega_{i_E}}{2}Z_{i_E}, \\ 
H_X &= \sum_{i_S} (g_x X_{i_S}), \\
H_{SE} &= \sum_{\langle i_S,i_E\rangle} O_{i_S}\otimes X_{i_E}.
\end{aligned}
$$

Within each Trotter step, the Trotterization can be broken down into the following sequence: 
$$e^{-iH(t)\delta_t} \approx e^{-i \alpha f(t) H_{SE} \delta_t / 2} e^{-i H_X \delta_t / 2} e^{-i H_Z \delta_t} e^{-iH_X \delta_t /2} e^{-i \alpha f(t) H_{SE} \delta_t / 2}$$

where $\delta_t$ = `dt` $= T/N_{\rm Trot}$ is the full Trotter step.

*Note that we are relying on the fact that $f(t)$ (and hence $H(t)$) is essentially constant in time over the course of the evolution.*

One more quick word of caution before you begin: the rotation gates `rzz`, `rx`, etc. introduce a factor of 1/2 within the phase argument. This can get very confusing when you also have to consider whether the evolution is by `dt` or `dt/2`, and whether there is a factor of 1/2 explicitly in the Hamiltonian term -- try and keep track of each one!

*Every `evolve_*` function appends gates to the circuit `circ` it is given (in place) and returns nothing. Re-running a cell only redefines the function: `H_tot_trot` creates a fresh `QuantumCircuit` every time it is called, so gates never accumulate across runs. Gates accumulate only if you call a function twice on the same circuit object.*

Following this Trotterization sequence, let's start by coding the evolution under $H_{SE}$. To do so, we need to specify the System-Environment edges that are included in the sum. This is done in the helper function `SE_Coupling`, which returns, for each Environment qubit $n = 0, \dots, N_E-1$ (virtual index $N_S+n$), the index of the System qubit it couples to. These are the virtual indices of Part 2.1 (System qubits $0,\dots,N_S-1$), **not** the physical qubits of the hardware layout in Part 2.5. (This is trivial for the small-scale system, but will be less trivial for larger systems where $N_S > N_E$ and some System-Environment couplings need to be sampled randomly.)

<div class="alert alert-block alert-success">

✏️ **Your task.** Fill in the `(2,2)` entry of `SE_Coupling`.

</div>

In [ ]:
def SE_Coupling(Lx,Ly):
    # returns a list of system qubits -- one for each environment qubit -- to establish the system-environment couplings for a particular application of Phi (virtual indices, not the hardware layout).
    edge_set_dict = {(2,2): ...,  # TODO: for each Environment qubit n, the (virtual) index of the System qubit it couples to
                     }
    return edge_set_dict[(Lx,Ly)]

Now we have established the <span style="color:darkgreen">System</span>-<span style="color:red">Environment</span> couplings. Given some `SE_couplings` returned from the `SE_Coupling` function, the next function executes the evolution under $\alpha f(t) H_{SE}$ for time `dt/2`. Note that you can implement any two-qubit Pauli using an `rzz` gate by sandwiching the gate between single-qubit gates that transform the basis.

<div class="alert alert-block alert-warning">

💡 **Hint: gates, basis changes and angles**

- **Angles.** Qiskit's rotation gates include a factor 1/2: $R_X(\theta)=e^{-i\theta X/2}$, $R_Z(\theta)=e^{-i\theta Z/2}$ and $R_{ZZ}(\theta)=e^{-i\theta Z\otimes Z/2}$. To implement $e^{-i\,c\,P\,t}$ (coefficient $c$, Pauli term $P$, evolution time $t$), use $\theta = 2ct$.
- **Evolution time per function.** Each `evolve_*` function receives the full Trotter step `dt`. Following the Trotter sequence above, `evolve_HSE` and `evolve_HX` evolve for `dt/2`; `evolve_HZ` evolves for the full `dt`.
- **Basis changes.** `rzz` implements $Z\otimes Z$. On the System qubit, the circuit $V$, `rzz`, $V^\dagger$ implements $P_S\otimes Z$ when $V^\dagger Z V = P_S$, where $P_S \in \{X, Y, Z\}$ is the jump operator:
    - **System X:** $V = H$, since $HZH = X$.
    - **System Y:** $V = HS^\dagger$, since $SHZHS^\dagger = Y$.
    - **System Z:** $V = I$ (no basis change).
    - **Environment:** the operator is always $X_E$, so $V = H$ on the Environment qubit.

  Operator products act right to left: $V = HS^\dagger$ means `sdg` first, then `h`, in the circuit.

- **Signs.** For a negative coupling, insert `x` gates on the **Environment qubit immediately before and immediately after `rzz`**, *inside* the Environment Hadamards. Since $X_E\,(Z_S\otimes Z_E)\,X_E=-Z_S\otimes Z_E$ (from $XZX=-Z$),

  $$X_E\, R_{ZZ}(\theta)\, X_E=R_{ZZ}(-\theta).$$

  These X gates change the **sign** of the interaction without changing the chosen Pauli operators. Their position is important: the sign flip happens **after** the initial basis changes and **before** they are undone.

</div>

<div class="alert alert-block alert-success">

✏️ **Your task.** Complete `evolve_HSE` to evolve under $\alpha f(t)H_{SE}$ for time `dt/2`, using the comments in the cell as a guide.

</div>

In [ ]:
def evolve_HSE(circ,            # QuantumCircuit object
               NS,              # number of System qubits
               NE,              # number of Environment qubits
               dt,              # full Trotter step size (this function evolves for dt/2)
               couplings,       # list of possible jump operators ['X', 'Y', 'Z']
               random_coupling, # list of NE randomly sampled indices (between 0 and len(couplings) - 1)
               random_sign,     # list of NE randomly sampled jump operator signs (1 or -1)
               SE_couplings,    # list of which system qubit is coupled to each environment qubit 0:NE-1
               f_val,           # value of f(t) for this particular step
               alpha,           # System-Environment coupling strength
               ):

    # Appends gates to circ in place; returns nothing.

    for n in range(NE):
        # Environment qubit NS+n couples to System qubit SE_couplings[n]
        system_site = SE_couplings[n]

        # The Environment operator is always X: basis change so the rzz acts as X on Environment qubit NS+n
        ...  # TODO: X-basis change on Environment qubit n+NS

        if (couplings[random_coupling[n]]) == 'X':
            # Basis change so the rzz acts as X on the System qubit (maps X-basis states onto Z-basis states)
            ...  # TODO: X-basis change on system_site
            if random_sign[n] == -1:
                # Introduce the sign on the Environment qubit using the Pauli identity XZX = -Z
                ...  # TODO: X on Environment qubit n+NS
                ...  # TODO: rzz between system_site and n+NS (angle from f_val, dt and alpha; see the hint)
                ...  # TODO: X on Environment qubit n+NS again
            else:
                ...  # TODO: rzz between system_site and n+NS (angle from f_val, dt and alpha; see the hint)
            # Undo the basis change (maps Z-basis states back onto X-basis states)
            ...  # TODO: undo the X-basis change on system_site
        elif (couplings[random_coupling[n]]) == 'Y':
            # Basis change so the rzz acts as Y on the System qubit (maps Y-basis states onto Z-basis states)
            ...  # TODO: Y-basis change on system_site (two gates)
            if random_sign[n] == -1:
                # Introduce the sign on the Environment qubit using the Pauli identity XZX = -Z
                ...  # TODO: X on Environment qubit n+NS
                ...  # TODO: rzz between system_site and n+NS (angle from f_val, dt and alpha; see the hint)
                ...  # TODO: X on Environment qubit n+NS again
            else:
                ...  # TODO: rzz between system_site and n+NS (angle from f_val, dt and alpha; see the hint)
            # Undo the basis change (maps Z-basis states back onto Y-basis states)
            ...  # TODO: undo the Y-basis change on system_site (two gates, in reverse order)
        elif (couplings[random_coupling[n]]) == 'Z':
            # Z coupling: no basis change on the System qubit
            if random_sign[n] == -1:
                # Introduce the sign on the Environment qubit using the Pauli identity XZX = -Z
                ...  # TODO: X on Environment qubit n+NS
                ...  # TODO: rzz between system_site and n+NS (angle from f_val, dt and alpha; see the hint)
                ...  # TODO: X on Environment qubit n+NS again
            else:
                ...  # TODO: rzz between system_site and n+NS (angle from f_val, dt and alpha; see the hint)

        # Undo the basis change on Environment qubit NS+n
        ...  # TODO: undo the X-basis change on Environment qubit n+NS

The next evolution in the Trotter sequence is the evolution under $H_X$ for time `dt/2`.

<div class="alert alert-block alert-success">

✏️ **Your task.** Implement this evolution in `evolve_HX` in the cell below -- *it should be much simpler than the previous evolution function!*

</div>

In [ ]:
def evolve_HX(circ, # QuantumCircuit object
              NS,   # number of system qubits
              dt,   # full Trotter step size (this function evolves for dt/2)
              gx,   # strength of transverse field
              ):

    # Add gates to the circuit to perform the evolution e^{-i H_X dt/2}.
    # Appends gates to circ in place; returns nothing.
    '''
    TODO: one rx on every System qubit implementing e^{-i H_X dt/2}
    '''
    ...

Finally, we have to consider evolution under $H_{Z}$. There are three different components of this: the single-qubit $Z$ terms acting on the <span style="color:darkgreen">System</span>, the single-qubit $Z$ terms acting on the <span style="color:red">Environment</span>, and the two-qubit $ZZ$ terms acting on the <span style="color:darkgreen">System</span> along the edges of the kagome lattice. This is the central evolution of the 2nd-order Suzuki-Trotter sequence, so evolve for the full `dt`.

Until now, we have not had to worry about optimizing the order of terms within a particular subgroup, because each term has acted on disjoint qubits. However, there are up to 4 $ZZ$ terms on a single qubit, and the ordering of these terms has a great impact on the final circuit depth. A highly optimized helper function `evolve_ZZ` is provided that implements the optimal ordering of the $ZZ$ terms for a particular lattice size.

<div class="alert alert-block alert-success">

✏️ **Your task.** Complete the `evolve_HZ` function to also perform the single-qubit $Z$ evolution of the <span style="color:darkgreen">System</span> and <span style="color:red">Environment</span>.

</div>

In [ ]:
from utils.gibbs_circuit_gen_utilities import evolve_ZZ

def evolve_HZ(circ,     # QuantumCircuit object
              NS,       # number of system qubits
              NE,       # number of environment qubits
              Lx,       # size of the horizontal dimension of the kagome lattice
              Ly,       # size of the vertical dimension of the kagome lattice
              dt,       # full Trotter step size (this function evolves for the full dt)
              gz,       # strength of longitudinal field
              omega_n,  # randomly sampled Bohr frequencies
              ):

    # evolve_ZZ (provided) applies the System ZZ terms for the full step dt.
    evolve_ZZ(circ,dt,Lx,Ly)

    # Below, add the single-qubit Z terms: e^{-i (H_E + sum_i g_z Z_i) dt}.
    # Appends gates to circ in place; returns nothing.
    '''
    TODO: one rz on each Environment qubit NS+n (from H_E = -sum_n omega_n/2 Z_n) and one rz on each
          System qubit (from g_z Z_i), each for the full step dt.
    '''
    ...

Now that we have all the elements for the Trotterization, we can piece them together.

<div class="alert alert-block alert-success">

✏️ **Your task.** Piece the evolution functions together in `one_Trot_step`, which is then used by `H_tot_trot`. Pass the same full step `dt` to every function: `evolve_HSE` and `evolve_HX` already evolve for `dt/2` internally.

</div>

In [ ]:
def one_Trot_step(circ,             # QuantumCircuit object
                  NS,               # number of system qubits
                  NE,               # number of environment qubits
                  Lx,               # size of horizontal dimension of the kagome lattice
                  Ly,               # size of vertical dimension of the kagome lattice
                  dt,               # full Trotter step size
                  couplings,        # list of possible jump operators ['X', 'Y', 'Z']
                  random_coupling,  # list of NE randomly sampled indices (between 0 and len(couplings) - 1)
                  random_sign,      # list of NE randomly sampled jump operator signs (1 or -1)
                  SE_couplings,     # list of which system qubit is coupled to each environment qubit 0:NE-1
                  f_val,            # value of f(t) for this particular step
                  alpha,            # system-environment coupling strength
                  gx,               # strength of transverse field
                  gz,               # strength of longitudinal field
                  omega_n,          # randomly sampled Bohr frequencies
                  ):

    # Use your evolve functions to implement a second-order Trotter step of the total Hamiltonian.
    # Appends gates to circ in place; returns nothing.
    '''
    TODO: call your evolve_* functions in the second-order Trotter order from Part 2.7
    '''
    ...


# Circuits that implement time evolution under H_alpha. f(t) is chosen to be a step function [0, t_tot].
# B_E is always X, and A_S is randomly drawn from {+/-X, +/-Y, +/-Z}. System qubits are on 0...NS-1, Bath qubits are NS,...(NS+NE)-1
def H_tot_trot(omega_n, # randomly sampled Bohr frequencies
               alpha,   # system-environment coupling strength
               nTrot,   # number of Trotter steps
               t_tot,   # total evolution time
               gx,      # transverse field strength
               gz,      # longitudinal field strength
               Lx,      # size of horizontal dimension of the kagome lattice
               Ly,      # size of vertical dimension of the kagome lattice
               NS,      # number of System qubits
               ):
    
    SE_couplings = SE_Coupling(Lx,Ly)
    NE = len(SE_couplings)
    circ = QuantumCircuit(NS+NE)
    # Trotter step size
    dt = t_tot/nTrot;sigma=1e2
    gauss = [np.exp(-x**2/t_tot**2/(4*sigma**2)) for x in np.arange(-t_tot/2 + t_tot/(2*nTrot),t_tot/2,t_tot/nTrot)]
    gauss = gauss/ np.sqrt( dt * np.sum( np.array(gauss)**2 ) )
    # Choosing the O_i (and its sign)
    couplings = ["X","Y","Z"] 
    random_coupling = np.random.randint(0, len(couplings), size=NE)
    random_sign =  np.random.choice([-1, 1], size=NE)
    for step in range(nTrot):
        f_val = gauss[step]  
        
        one_Trot_step(circ, NS, NE, Lx, Ly, dt, couplings, random_coupling, random_sign, SE_couplings, f_val, alpha, gx, gz, omega_n)

    return circ

<div class="alert alert-block alert-info">

**Check your Trotterization implementation below (11.5 pts total).**

</div>

In [ ]:
grade_lab_thermal_states_ex2_7(SE_Coupling, evolve_HSE, evolve_HX, evolve_HZ, one_Trot_step)

### 2.8 Build one circuit and look inside it

In this section, we turn the Gibbs-sampling circuit into a form that IBM Quantum hardware can run. We will compile the logical circuit, prepare randomized versions with Samplomatic, and add timing-aware error suppression.

#### A. Transpile the logical circuit for the QPU

**Transpilation** converts the circuit into the backend's instruction set (an *ISA circuit*). In `compile_to_isa(...)`, we use the physical layout chosen earlier and `optimization_level=0` to retain the intended Trotter structure. `RemoveGlobalPhasePass` removes a phase that has no effect on measurement outcomes.

The following cell compiles a small example with `nSteps=2`, including a bath-reset window. Comparing the logical and ISA operation counts helps us see what transpilation changed.

API reference: [`generate_preset_pass_manager`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.transpiler.generate_preset_pass_manager).

<div class="alert alert-block alert-success">

✏️ **Your task.** Complete `compile_to_isa`: build a preset pass manager for `backend` with the physical `layout` as the initial layout and the optimization level given above, then run it on the logical circuits.

</div>

In [ ]:
from utils.gibbs_circuit_generator import build_logical_circuits
from utils.gibbs_transpilation_utilities import RemoveGlobalPhasePass
from qiskit.transpiler import PassManager, generate_preset_pass_manager


def compile_to_isa(logical_circuits, backend, layout):
    """Backend-aware transpilation (same settings as the original generator)."""
    pm = generate_preset_pass_manager(
        backend=backend,
        initial_layout=...,  # TODO: physical-qubit layout
        optimization_level=...,  # TODO: the level given in the text above (keeps the hand-built Trotter structure)
    )
    isa_circuits = ...  # TODO: run the pass manager on logical_circuits
    return PassManager([RemoveGlobalPhasePass()]).run(isa_circuits)


training_logical = training_isa = None
if backend is not None:
    training_reset_dt = int(backend.properties().gate_length("reset", layout[0]) / 4e-9)
    training_logical = build_logical_circuits(
        Lx, Ly, 2, 100.0, [0.0], BASE_SEED,
        training_reset_dt, init_environment, H_tot_trot, hardware_layouts,
        alpha, nTrot, t_tot, gx, gz, init_system=init_system
    )[0]
    training_isa = compile_to_isa([training_logical], backend, layout)[0]
    print("Logical:", training_logical.count_ops())
    print("ISA:", training_isa.count_ops())
else:
    print("Connect to a backend to run the backend-aware transpilation demonstration.")


<div class="alert alert-block alert-info">

**Check your `compile_to_isa` implementation below (1.5 pts).**

</div>

In [ ]:
grade_lab_thermal_states_ex2_8a(compile_to_isa)

#### B. Prepare Pauli twirling with Samplomatic

Real gates can have systematic errors that accumulate across a circuit. **Pauli twirling** uses random Pauli operations ($I$, $X$, $Y$, or $Z$), together with compensating changes, so that each randomized implementation performs the **same ideal computation**. Averaging over implementations can make some coherent errors behave more like stochastic noise.

The following code first uses `generate_boxing_pass_manager(...)` to mark the gates and measurements to be randomized. We use **local Pauli twirling** (`twirling_group="local_pauli"`) and twirl only the qubits that are **active** in each layer (`twirling_strategy="active"`; the `active_accum` and `active_circuit` strategies would also twirl qubits that are idle in a layer), while retaining the barrier handling needed for the reset windows. Samplomatic's `build(...)` then returns two matching objects:

- A **template**: a parameterized quantum circuit whose randomized choices are filled in later.
- A **Samplex**: a classical description of how to sample those choices and track any measurement-bit corrections.

The boxed circuit identifies *where* randomization may occur. It is not itself a set of sampled circuits. See the [Samplomatic boxing guide](https://quantum.cloud.ibm.com/docs/en/addons/samplomatic/guides/transpiler).

<div class="alert alert-block alert-success">

✏️ **Your task.** Complete `box_and_build`: choose the twirling group and strategy described above, run the boxing pass manager on the ISA circuits, and call Samplomatic's `build(...)` on each boxed circuit to get its template and Samplex.

</div>

In [ ]:
from samplomatic import build
from samplomatic.transpiler import generate_boxing_pass_manager


def box_and_build(isa_circuits, *, return_boxed=False):
    """Create boxed circuits, template circuits and their matching samplexes."""
    boxing_pm = generate_boxing_pass_manager(
        enable_gates=True,
        enable_measures=True,
        twirling_group=...,  # TODO: the local Pauli twirling group (see the text above)
        twirling_strategy=...,  # TODO: twirl only the active qubits of each layer (see the text above)
        decomposition="rzrx",
        remove_barriers="after_stratification",
    )
    boxed_circuits = ...  # TODO: apply boxing to isa_circuits

    templates = []
    samplexes = []
    for circ in boxed_circuits:
        template, samplex = ...  # TODO: build the template and samplex from circ
        templates.append(template)
        samplexes.append(samplex)

    if return_boxed:  # used only for the teaching demonstration
        return templates, samplexes, boxed_circuits
    return templates, samplexes


training_boxed = training_template = training_samplex = None
if training_isa is not None:
    _templates, _samplexes, _boxed = box_and_build(
        [training_isa], return_boxed=True
    )
    training_boxed = _boxed[0]
    training_template = _templates[0]
    training_samplex = _samplexes[0]
    print("Boxed:", training_boxed.count_ops())
    print("Template parameters:", training_template.num_parameters)
else:
    print("Run the backend-aware transpilation example first.")


<div class="alert alert-block alert-info">

**Check your `box_and_build` implementation below (1.5 pts).**

</div>

In [ ]:
grade_lab_thermal_states_ex2_8b(box_and_build)

#### C. Build parameter sweeps for initial states and readout bases

We want to repeat a circuit with different initial <span style="color:darkgreen">System</span> bitstrings, different Pauli twirls, and two measurement bases. The template circuit still contains **symbolic parameters**: `init[0]`, …, `init[NS-1]` from `init_system` (Part 2.6) and `post_rot`, the final readout rotation. Before running, every parameter needs a number. `samplex_parameter_values(...)` builds all these numbers at once, in an array with axes

$$
(N_{\mathrm{init}},\;N_{\mathrm{twirls}},\;2,\;N_{\mathrm{parameters}}).
$$

- **Initial state (`init[n]`).** A $\beta=0$ <span style="color:darkgreen">System</span> state is a uniformly random bitstring: as shown in Part 2.6, $\rho_{S,0}=I/2^{N_S}$ gives every bitstring the same probability, which is the same as choosing each bit independently as 0 or 1 with probability 1/2. Draw these bits with the seeded generator `rng`, so the same `seed` always gives the same sweep. The angle `init[n]` is $0$ for bit 0 ($|0\rangle$) and $\pi$ for bit 1 ($|1\rangle$).
- **Same bitstring for all twirls and both bases.** Each initial bitstring (index `ii`) is drawn **once** and reused for all its twirls and both readout bases.
- **Readout basis (`post_rot`).** Basis index 0 uses `post_rot` $=0$ ($Z$ readout); index 1 uses `post_rot` $=\pi/2$ ($X$ readout). These are the entries of `post_rot_values`.
- **Column order.** The columns must follow the order of `samplex.parameters`, which need not be `init[0], init[1], …`. The dictionary `idx` maps each parameter name to its column.

<div class="alert alert-block alert-success">

✏️ **Your task.** Complete `samplex_parameter_values` following the rules above.

</div>

In [ ]:
def samplex_parameter_values(samplex, NS, ninit, ntwirls, seed,
                             post_rot_values=(0, np.pi / 2)):
    names = [p.name for p in samplex.parameters]
    idx = {name: i for i, name in enumerate(names)}
    rng = np.random.default_rng(seed)
    values = np.zeros((..., ..., ..., len(names)))  # TODO: the first three axes: number of initial states, number of twirls, number of readout bases
    for ii in range(ninit):
        bits = ...  # TODO: NS random bits (each 0 or 1 with probability 1/2), drawn with rng
        for tt in range(ntwirls):
            for basis, angle in enumerate(post_rot_values):
                for n in range(NS):
                    values[ii, tt, basis, idx[f"init[{n}]"]] = ...  # TODO: angle for init[n]: 0 if bits[n] is 0, pi if bits[n] is 1
                values[ii, tt, basis, idx["post_rot"]] = ...  # TODO: readout angle for this basis (from post_rot_values)
    return values

The following small test uses a stand-in Samplex parameter list to check the sweep's shape and consistency. 

In [ ]:
class _ParameterPreview:
    def __init__(self, name): self.name = name
class _SamplexPreview:
    parameters = [_ParameterPreview(f"init[{n}]") for n in range(3)] + [_ParameterPreview("post_rot")]

_preview = samplex_parameter_values(_SamplexPreview(), 3, 2, 3, seed=42)
assert _preview.shape == (2, 3, 2, 4)
assert np.all(_preview[:, :, 0, :3] == _preview[:, :, 1, :3])
assert np.all(_preview[:, :, :, :3] == _preview[:, :1, :, :3])
assert np.all(_preview[:, :, 0, 3] == 0)
assert np.all(_preview[:, :, 1, 3] == np.pi / 2)
print("Samplex sweep checks passed.")

<div class="alert alert-block alert-info">

**Check your `samplex_parameter_values` implementation below (1.5 pts).**

</div>

In [ ]:
grade_lab_thermal_states_ex2_8c(samplex_parameter_values)

#### D. Inspect and sample the Samplex locally

Before using a Samplex on hardware, we can inspect its **inputs** (values we must provide) and **outputs** (sampled parameters and measurement corrections). The next cell binds one sweep entry, checks `fully_bound`, and requests two randomizations **on the classical computer**.

`inputs().bind(parameter_values=...)` takes the values for **one** sweep entry: a 1D array with one value per Samplex parameter, i.e. one `[init, twirl, basis]` entry of the array from Part 2.8C. `sample(inputs, num_randomizations=...)` then draws that many random twirls of the bound circuit.

Measurement twirling may flip a recorded bit $b$. If the Samplex returns correction bit $f$, the logical result is $b\oplus f$. These corrections are needed later when estimating observables; they are distinct from the extra `meas_ps` register used for post-selection. For more background, see the [Samplex inputs and outputs guide](https://quantum.cloud.ibm.com/docs/en/addons/samplomatic/guides/samplex-io).

<div class="alert alert-block alert-success">

✏️ **Your task.** Bind the first sweep entry and draw two randomizations locally.

</div>

In [ ]:
sampled = None
if training_samplex is not None:
    print(training_samplex)
    print("Input specs:", training_samplex.inputs().specs)
    print("Output specs:", training_samplex.outputs().specs)
    print("Samplex parameters:", [p.name for p in training_samplex.parameters])
    local_values = samplex_parameter_values(training_samplex, NS, 1, 1, BASE_SEED)
    print("Local parameter array shape:", local_values.shape)
    local_inputs = training_samplex.inputs().bind(parameter_values=...)  # TODO: the first sweep entry of local_values (init 0, twirl 0, basis 0), one value per parameter
    if local_inputs.fully_bound:
        sampled = training_samplex.sample(..., num_randomizations=...)  # TODO: the bound inputs and the number of randomizations (2)
        print("Locally sampled template parameter shape:", sampled["parameter_values"].shape)
        for spec in training_samplex.outputs().get_specs("flips"):
            print("Flip output:", spec.name, sampled[spec.name].shape)
    else:
        print("Additional inputs are required by this installed Samplomatic version:", local_inputs)
else:
    print("Build the Samplomatic samplex first.")

#### E. Finalize the circuit and assemble the generator

The Samplomatic template still needs a few hardware-specific steps. `finalize_templates(...)` adds **`xslow`-based error checks** (recorded in `meas_ps` for Part 3), restores barriers around bath resets, and schedules operations using backend gate durations. It then applies **dynamical decoupling (DD)**: pairs of $X$ pulses in suitable idle periods, intended to suppress unwanted evolution while other qubits are being reset. Unlike Pauli twirling, DD acts during *idle time*.

The scheduling part of `finalize_templates` is a [`PassManager`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.transpiler.PassManager) with three passes, which run in the order they appear in the list:

1. `RestoreResetBarriers(barrier_qubits=layout)` is provided in `utils/restore_reset_barriers.py` (imported at the top of the cell; its docstring explains it) and is already filled in. Samplomatic's boxing step removes all barriers, including the ones around the reset windows; this pass puts them back, spanning every qubit of `layout`.
2. [`ALAPScheduleAnalysis`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.transpiler.passes.ALAPScheduleAnalysis) schedules every instruction as late as possible, using the backend's `instruction_durations`.
3. [`PadDynamicalDecoupling`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling) fills the idle windows with the DD sequence.

The same cell defines `generate_gibbs_circuit(...)`, which brings together **compilation, Samplomatic, and finalization** into one function for the later experiments. See the [IBM dynamical-decoupling guide](https://quantum.cloud.ibm.com/docs/en/guides/dynamical-decoupling-pass-manager).

<div class="alert alert-block alert-success">

✏️ **Your task.** Fill in the two missing entries of `scheduling_pm`: the ALAP scheduling pass and the DD sequence.

</div>

In [ ]:
from qiskit.circuit.library import XGate
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDynamicalDecoupling
from utils.post_selection_with_resets import AddPostSelectionMeasuresWithResets
from utils.restore_reset_barriers import RestoreResetBarriers

POST_ROT_VALUES = [[0, np.pi / 2]]


def finalize_templates(templates, backend, layout, post_selection=True,
                       *, return_pre_dd=False):
    """Post-selection and scheduled DD, with the original order and settings."""
    if post_selection and "xslow" not in backend.target:
        raise ValueError("'xslow' is not in backend.target -- load the backend with a "
                         "calibration_id that provides it")

    if post_selection:
        ps_pm = PassManager([AddPostSelectionMeasuresWithResets("xslow")])
        templates = [ps_pm.run(template) for template in templates]

    # Snapshot after post-selection but before barrier restoration/scheduling/DD.
    pre_dd = [c.copy() for c in templates] if return_pre_dd else None
    scheduling_pm = PassManager([
        RestoreResetBarriers(barrier_qubits=layout),  # provided: restores the reset-window barriers removed by boxing
        ...,  # TODO: ALAPScheduleAnalysis with the backend's instruction durations
        PadDynamicalDecoupling(
            durations=backend.instruction_durations,
            dd_sequence=...,  # TODO: a two-pulse X-X sequence, as a list of gates (XGate is imported above)
            skip_reset_qubits=True,
        ),
    ])
    scheduled = scheduling_pm.run(templates)
    return (scheduled, pre_dd) if return_pre_dd else scheduled


def generate_gibbs_circuit(backend, Lx, Ly, nSteps, init_environment, H_tot_trot,
                           hardware_layouts, ALPHA, NTROT, T_TOT, GX, GZ,
                           beta=100, noise=[0.0], seed=42, post_selection=True,
                           init_system=None, *, return_pre_dd=False):
    """One randomization: actual execution uses the Part 2.8 notebook functions.

    Returns (scheduled_templates, samplexes, post_rot_values), one per noise factor.
    When return_pre_dd=True, also returns pre-DD templates for diagnostics only.
    """
    layout = hardware_layouts(Lx, Ly)
    reset_duration = int(backend.properties().gate_length("reset", layout[0]) / 4e-9)

    logical_circuits = build_logical_circuits(
        Lx, Ly, nSteps, beta, list(noise), seed, reset_duration,
        init_environment, H_tot_trot, hardware_layouts,
        ALPHA, NTROT, T_TOT, GX, GZ, init_system=init_system,
    )
    isa_circuits = compile_to_isa(logical_circuits, backend, layout)
    templates, samplexes = box_and_build(isa_circuits)
    final = finalize_templates(
        templates, backend, layout, post_selection=post_selection,
        return_pre_dd=return_pre_dd,
    )
    if return_pre_dd:
        scheduled, pre_dd = final
        return scheduled, samplexes, POST_ROT_VALUES, pre_dd
    return final, samplexes, POST_ROT_VALUES


demo_templates = demo_pre_dd = None
if backend is not None:
    # Coldest beta, deepest depth, physical noise only. Same code path as QPU submission.
    demo_templates, demo_samplexes, demo_post_rot, demo_pre_dd = generate_gibbs_circuit(
        backend, Lx, Ly, 7, init_environment, H_tot_trot, hardware_layouts,
        ALPHA=alpha, NTROT=nTrot, T_TOT=t_tot, GX=gx, GZ=gz,
        beta=100.0, seed=BASE_SEED, init_system=init_system,
        return_pre_dd=True,
    )

    circ = demo_templates[0]
    part2_two_qubit_depth = circ.depth(
        filter_function=lambda inst: inst.operation.name in ("cz", "rzz")
    )
    reset_barriers = [
        inst.operation.label for inst in circ.data
        if inst.operation.name == "barrier"
        and (inst.operation.label or "").startswith(("pre_reset", "post_reset"))
    ]
    print("templates returned (one per noise factor):", len(demo_templates))
    print("two-qubit (cz/rzz) depth               :", part2_two_qubit_depth)
    print("dynamical-decoupling X pulses          :", circ.count_ops().get("x", 0))
    print("reset-window barrier labels            :", reset_barriers)
    print("classical registers                    :", [creg.name for creg in circ.cregs])
else:
    print("Backend not connected -> skipping circuit build. (Set CONNECT_TO_QPU=True.)")


<div class="alert alert-block alert-info">

**Check your `finalize_templates` implementation below (1.5 pts).**

</div>

In [ ]:
grade_lab_thermal_states_ex2_8e(finalize_templates)

#### F. Check the compiled circuit

The demo at the end of Part E's code cell reports the template count, two-qubit (`cz`/`rzz`) depth, DD pulses, reset barriers, and classical registers. **Two-qubit depth counts gate layers, not QPU execution time.** The next cell compares operation counts **before and after DD**, so you can check whether the scheduling pass changed the circuit as intended. The `nSteps` setting is not itself a count of physical `reset` instructions: with `nSteps=7` you will see six reset windows (`pre_reset_0` … `post_reset_5`). See the note just before Part 2.9.

In [ ]:
if demo_templates is not None:
    pre_dd = demo_pre_dd[0]
    post_dd = demo_templates[0]
    print("Before DD operation counts:", pre_dd.count_ops())
    print("After  DD operation counts:", post_dd.count_ops())
    print("X operations before / after:",
          pre_dd.count_ops().get("x", 0), post_dd.count_ops().get("x", 0))
    print("Delays before / after:",
          pre_dd.count_ops().get("delay", 0), post_dd.count_ops().get("delay", 0))
    print("Reset operations:", post_dd.count_ops().get("reset", 0))
else:
    print("Build the demonstration template to inspect DD.")

<div class="alert alert-block alert-info">

**Check your 2-qubit gate depth below (1 pt).**

</div>

In [ ]:
grade_lab_thermal_states_ex2_8f(part2_two_qubit_depth)

#### Conceptual check (feedback only, 0 points)

**q1.** Why does `box_and_build` use `twirling_group="local_pauli"` for these circuits?
- (a) It is the fastest twirling group to sample.
- (b) The circuits contain fractional (non-Clifford) `rzz` gates; in the installed Samplomatic version, `local_pauli` is the only twirling group that can twirl them (the others raise a `BuildError`).
- (c) It removes the need for measurement twirling.
- (d) Dynamical decoupling requires it.

**q2.** What do the `measurement_flips.meas` bits returned by the Samplex do?
- (a) They flag shots to discard during post-selection.
- (b) They mark leakage events detected by the `xslow` check.
- (c) They undo the randomized measurement twirl: the logical bit is $b \oplus f$.
- (d) They encode which readout basis ($Z$ or $X$) was used.

**q3.** Why does `compile_to_isa` use `optimization_level=0`?
- (a) To keep the hand-built Trotter structure (and its two-qubit depth) instead of letting the transpiler re-synthesize it.
- (b) Higher optimization levels do not support fractional gates.
- (c) It makes the transpilation randomized, like twirling.
- (d) Level 0 inserts dynamical decoupling automatically.

**q4.** Why are the reset-window barriers restored *before* scheduling and dynamical decoupling?
- (a) Post-selection needs the barriers to find the `meas_ps` register.
- (b) Samplomatic needs barriers to build a template.
- (c) Barriers reduce the two-qubit depth.
- (d) They fence each reset window, so DD refocuses a bounded idle interval on the System qubits.

<div class="alert alert-block alert-success">

✏️ **Your task.** Answer each question with the letter (a, b, c or d) of the best option in `part2p8_answers` in the cell below.

</div>

In [ ]:
part2p8_answers = {"q1": "", "q2": "", "q3": "", "q4": ""}  # TODO: fill in a, b, c or d

In [ ]:
grade_lab_thermal_states_ex2_8_concepts(part2p8_answers)

#### G. Package a training `QuantumProgram`

A `QuantumProgram` groups the circuits and sampling instructions that will eventually be sent to the QPU. Here, `append_samplex_item(...)` pairs **one template with its matching Samplex** and supplies the parameter sweep. The shape `(N_INIT, N_TWIRLS, 2)` describes the initial-state, twirl, and measurement-basis axes; `shots=64` requests 64 measurements **per sweep entry**.

This is only a training program: **creating it does not submit a job**. Hardware execution begins in Part 2.9 using `Executor`. See the [Executor quickstart](https://quantum.cloud.ibm.com/docs/en/guides/get-started-with-executor).

<div class="alert alert-block alert-success">

✏️ **Your task.** Add the training template and its Samplex to `training_program`, with the sweep `training_values` and its shape.

</div>

In [ ]:
from qiskit_ibm_runtime.quantum_program import QuantumProgram
training_program = None
if training_samplex is not None:
    training_values = samplex_parameter_values(training_samplex, NS, 1, 2, BASE_SEED)
    training_program = QuantumProgram(shots=64)
    training_program.append_samplex_item(
        ...,  # TODO: the training template circuit
        samplex=...,  # TODO: its matching Samplex
        samplex_arguments={"parameter_values": ...},  # TODO: the parameter sweep built above
        shape=(..., ..., ...),  # TODO: (inits, twirls, bases) of training_values
    )
    print("Training configurations:", 1 * 2 * 2)
else:
    print("Construct the samplex before creating a training QuantumProgram.")

<div class="alert alert-block alert-info">

**Check your Samplex sampling (2.8D) and training `QuantumProgram` below (1.5 pts).**

</div>

In [ ]:
grade_lab_thermal_states_ex2_8d_g(training_program, training_template, training_samplex, sampled)

<div class="alert alert-block alert-warning">

***A note on the depth argument.***  
We pass the reset-depth value straight into the generator's `nSteps` argument (matching how the saved reference data was produced). Internally the builder does `nSteps - 1` physical bath resets, while the noiseless reference loaded in Part 2.2 counts the number of channel applications $N_{\rm resets}$. We keep this convention exactly and simply plot each QPU point at $x = N_{\rm resets}$, so the hardware points line up with the precomputed reference.

</div>

### 2.9 Submit the experiment

This is the first point at which the notebook can **consume QPU time**. All of Part 2.8 was circuit construction, inspection and local classical sampling. Here `submit_to_QPU(...)` constructs `QuantumProgram` objects and, **only if `RUN_QPU_SMALL=True`**, calls `Executor(mode=backend).run(program)` to launch hardware jobs.

**What is sent?** Each distinct **algorithmic randomization** (fresh bath frequencies, sampled couplings and associated circuit realization) becomes **one job** for a fixed pair $(\beta,N_{\rm resets})$. Inside the job, `append_samplex_item(...)` requests a sweep over:

1. `N_INIT` random initial $\beta=0$ <span style="color:darkgreen">System</span> bitstrings;
2. `N_TWIRLS` Pauli-twirled realizations of each initialization;
3. **2 readout bases:** $Z$ and $X$, needed to reconstruct the energy.

For the supplied settings, the sweep has $5\times10\times2=100$ entries **per job per noise setting**, with **64 shots per entry**. That is $6{,}400$ measurement shots per job for Part 2's single physical-noise setting. With `N_RANDOMIZATIONS=8`, the nominal total is $51{,}200$ shots **per temperature/reset configuration**. The small experiment has **two** bath temperatures ($\beta=0.25,100$) and **four** requested depth settings (`SMALL_RESETS=(0,1,3,7)`): this is eight configurations in all. If every configuration is submitted, the loop creates **64 jobs** and requests **409,600 shots** at the default settings. This arithmetic counts measurement attempts, *not* predicted QPU wall time; actual usage also depends on circuit duration, resets, device/runtime overhead, queue and backend limits.

**Why the two bases?** The measured observables reconstruct the kagome system energy density:

$$
\frac{E}{N_S}=\frac{1}{N_S}\left[\sum_{\langle i,j\rangle}\langle Z_iZ_j\rangle +g_x\sum_i\langle X_i\rangle +g_z\sum_i\langle Z_i\rangle\right].
$$

The **$Z$-basis** results give both $\langle Z_i\rangle$ and $\langle Z_iZ_j\rangle$; the **$X$-basis** results give $\langle X_i\rangle$. Within each returned job item, `meas` holds the randomized measured bits and `measurement_flips.meas` holds corrections to undo the measurement twirls. Keep the two arrays aligned: the processing in Part 2.11 explicitly applies the flips before computing expectation values. The extra `meas_ps` register is recorded for **Part 3 post-selection**; it is **not** used to filter the raw Part 2 energies.

**Understanding the four nested layers:** `N_RANDOMIZATIONS` counts separately generated Gibbs circuits/jobs; `N_INIT` counts initial-state parameter settings in each job; `N_TWIRLS` counts randomized dressings within each setting. `N_SHOTS` is the number of repeated physical measurements of each setting. These are not four names for the same operation. At `NOISE_AMP=None`, `noise=[0.0]` means **no *deliberate* noise amplification**, *not* that the hardware is noiseless. In Part 3 the helper instead includes multiple noise-amplified templates in a single program, using the same stochastic realization for fair ZNE comparison.

Run the cell below to establish `submit_to_QPU`, which builds the circuits and submits them through Executor, together with `retrieve_results`, which loads saved results or reads back completed jobs.

Further reading: [IBM Executor quickstart](https://quantum.cloud.ibm.com/docs/en/guides/get-started-with-executor), [Executor inputs and outputs](https://quantum.cloud.ibm.com/docs/en/guides/executor-input-output), [Executor broadcasting](https://quantum.cloud.ibm.com/docs/en/guides/executor-broadcasting), and [IBM Pauli twirling/error suppression](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques).


In [ ]:
import yaml

def submit_to_QPU(RUN_QPU, JOB_FILE, Lx, Ly, NS, BETAS, RESETS, BASE_SEED, N_RANDOMIZATIONS, N_SHOTS, N_INIT, N_TWIRLS, NOISE_AMP=None, ALPHA=alpha, NTROT=nTrot, T_TOT=t_tot, GX=gx, GZ=gz): 
    if not isinstance(BETAS, (list, tuple)):
        BETAS = [BETAS]
    if not isinstance(RESETS, (list, tuple)):
        RESETS = [RESETS]

    if RUN_QPU:
        if backend is None:
            raise RuntimeError("Set CONNECT_TO_QPU=True and run the connect cell first.")

        from qiskit_ibm_runtime import Executor
        from qiskit_ibm_runtime.quantum_program import QuantumProgram
        import gc

        small_job_ids = {}
        executor = Executor(mode=backend)

        for beta in BETAS:
            for nreset in RESETS:
                base_seed = BASE_SEED + 100 * nreset + int(beta)
                key = f"beta_{beta}_resets_{nreset}"
                if NOISE_AMP is None: 
                    noise = [0.0]
                    print("\nSubmitting", key)
                else: 
                    noise = list(NOISE_AMP)
                    print("\nSubmitting amplified", key, "  noise =", noise)

                jobs = {}
                for randomization in range(N_RANDOMIZATIONS):
                    instance_seed = base_seed + 30 * randomization
                    templates, samplexes, _ = generate_gibbs_circuit(
                        backend, Lx, Ly, nreset, init_environment, H_tot_trot, hardware_layouts,
                        ALPHA=ALPHA, NTROT=NTROT, T_TOT=T_TOT, GX=GX, GZ=GZ, 
                        beta=beta, noise=noise, seed=instance_seed, init_system=init_system)

                    program = QuantumProgram(shots=N_SHOTS)
                    for template, samplex in zip(templates, samplexes):
                        program.append_samplex_item(
                            template,
                            samplex=samplex,
                            samplex_arguments={"parameter_values": samplex_parameter_values(
                                samplex, NS, N_INIT, N_TWIRLS, seed=instance_seed)},
                            # axes: (init, twirl, basis); basis index 0 -> Z, index 1 -> X
                            shape=(N_INIT, N_TWIRLS, 2),
                        )

                    job = executor.run(program)
                    jobs[f"random_{randomization}"] = job.job_id()
                    print(f"  random_{randomization}: {job.job_id()}")
                    del templates, samplexes, program, job
                    gc.collect()

                small_job_ids[key] = {"beta": float(beta), "n_resets": int(nreset),
                                    "noise": noise, "n_init": int(N_INIT),
                                    "n_twirls": int(N_TWIRLS),
                                    "layout_key": [int(Lx), int(Ly)],
                                    "job_ids": jobs}

        with JOB_FILE.open("w") as f:
            yaml.safe_dump(small_job_ids, f, sort_keys=False)
        print("\nSaved job IDs to", JOB_FILE)
    else:
        print("RUN_QPU is False -> not submitting.")
        print("The existing IDs in", JOB_FILE, "will be reused by the retrieve cell.")
        

def retrieve_results(service, JOB_FILE, CACHE_FILE=None):
    # Reuse previously saved results if the cache file exists and isn't empty.
    if CACHE_FILE is not None and CACHE_FILE.exists():
        with CACHE_FILE.open("rb") as f:
            cached = pickle.load(f)
        if cached:
            print(f"Loaded {len(cached)} saved configuration(s) from {CACHE_FILE}.")
            return cached
        print(f"{CACHE_FILE} is empty -> retrieving from the QPU instead.")

    if service is None or not JOB_FILE.exists():
        print("No saved job file -> nothing retrieved.")
        return None

    with JOB_FILE.open("r") as f:
        job_cfgs = yaml.safe_load(f)

    # Check every job's status first, so this cell never blocks waiting on the QPU.
    jobs = {
        key: {rk: service.job(jid) for rk, jid in cfg["job_ids"].items()}
        for key, cfg in job_cfgs.items()
    }
    statuses = {
        (key, rk): job.status()
        for key, key_jobs in jobs.items() for rk, job in key_jobs.items()
    }
    counts = {s: list(statuses.values()).count(s) for s in sorted(set(statuses.values()))}
    print("Job status:", ", ".join(f"{s}={n}" for s, n in counts.items()))

    failed = [(key, rk) for (key, rk), s in statuses.items() if s in ("ERROR", "CANCELLED")]
    if failed:
        print("\nSome jobs failed or were cancelled -> resubmit them (RUN_QPU=True) before retrieving:")
        for key, rk in failed:
            job = jobs[key][rk]
            print(f"  {key} / {rk}: {job.job_id()} ({statuses[(key, rk)]}) {job.error_message() or ''}")
        return None

    n_done = counts.get("DONE", 0)
    if n_done < len(statuses):
        print(f"\nJobs are still running ({n_done}/{len(statuses)} done). "
              "Rerun this cell once all jobs are complete.")
        return None

    print(f"\nAll {len(statuses)} jobs are done running.")
    results = {}
    for key, key_jobs in jobs.items():
        print("Retrieving", key)
        results[key] = {rk: job.result() for rk, job in key_jobs.items()}

    print(f"\nResults retrieved successfully: {len(results)} configuration(s).")
    return results


The cell below is guarded by `RUN_QPU_SMALL`. When it runs, the job IDs are written to the YAML file once every job in the call has been submitted, so the analysis can be rerun later without resubmitting anything.

The job IDs are kept in one file:

- `SMALL_JOB_FILE`: the thermalization points $N_{\rm resets}=0,1,3,7$ for both bath temperatures.

With `RUN_QPU = False`, no new workload is submitted and any existing saved IDs can be reused.


In [ ]:
SMALL_JOB_FILE = Path(f"./job_info/qdc26_small_gibbs_job_ids_{BACKEND_NAME}.yaml")

# Part 2: physical-noise thermalization points.
submit_to_QPU(
    RUN_QPU=RUN_QPU_SMALL,
    JOB_FILE=SMALL_JOB_FILE,
    Lx=Lx, 
    Ly=Ly,
    NS=NS,
    BETAS=SMALL_BETAS,
    RESETS=SMALL_RESETS,
    BASE_SEED=BASE_SEED,
    N_RANDOMIZATIONS=N_RANDOMIZATIONS,
    N_SHOTS=N_SHOTS,
    N_INIT=N_INIT,
    N_TWIRLS=N_TWIRLS,
)

### 2.10 Retrieve the results

Retrieval is always safe to rerun: it only reads jobs that have already been submitted and never creates new QPU workloads.

If `submitted_results/part2_results.pkl` exists and is non-empty, it is loaded and no backend connection is needed. Otherwise this requires `CONNECT_TO_QPU = True`: the cell checks the status of every job, and if any job is still queued or running, it prints a status summary and returns without waiting; rerun it once all jobs are complete.


In [ ]:
part2_results = retrieve_results(
    service, SMALL_JOB_FILE, CACHE_FILE=Path('./submitted_results/part2_results.pkl')
)

### 2.11 From measurements to energy

Recall the *Bridge* section: the AFIM energy needs only **two measurement bases**.

- The **$Z$-basis** shots (basis index `0`) give both $\langle Z_i\rangle$ and $\langle Z_iZ_j\rangle$.
- The **$X$-basis** shots (basis index `1`) give $\langle X_i\rangle$.

Each retrieved result stores its counts in `result["meas"]`, with axes `(init, twirl, basis, shots, bits)`: one job bundles several random $\beta=0$ initializations, several Pauli twirls of each, and the two measurement bases. To turn shots into an expectation value, we use two tiny helpers that mirror the reference pipeline:

- `zstring(idxs, nbits)` builds the Pauli label with `Z` on the requested qubits and `I` elsewhere;
- `get_vals(...)` hands that operator, the chosen basis slice, and the measurement-flip / optional post-selection bookkeeping to `exp_vals`, and returns **one expectation value per (init, twirl)**.

We read $\langle X_i\rangle$ using the *same* `Z`-operator label but on the X-basis slice (`param_index=1`), because a basis rotation was applied before those shots were measured. The energy per <span style="color:darkgreen">System</span> qubit is then:

$$\frac{E}{N_S} = \frac{1}{N_S}\Big(\sum_{\langle i,j\rangle} \langle Z_i Z_j \rangle + g_x \sum_i \langle X_i \rangle + g_z \sum_i \langle Z_i\rangle \Big).$$

**Helper 1: `zstring(idxs, nbits)`** returns the Pauli label of length `nbits` with `Z` on the qubits in `idxs` and `I` elsewhere. Qiskit labels are little-endian: qubit 0 is the **rightmost** character. Example: `zstring([0, 2], 4) == "IZIZ"`.

**Helper 2: `get_vals(result, param_index, idxs, nbits, mask)`** computes $\langle Z\cdots Z\rangle$ on the qubits `idxs` from the shots of one basis, in three steps:
1. Slice `result["meas"]` (axes `(init, twirl, basis, shots, bits)`) at `basis = param_index`, keeping every other axis (the `measurement_flips` line in the cell shows the same slicing).
2. Build the observable `SparsePauliOp(zstring(idxs, nbits))`, then call [`exp_vals.get_measurement_bases`](https://quantum.cloud.ibm.com/docs/en/api/qiskit-addon-utils/exp-vals#get_measurement_bases) on it. It returns two things: a list of bases and a dictionary mapping each basis to the observables measured in it. `executor_expectation_values` needs the **dictionary**.
3. [`exp_vals.executor_expectation_values`](https://quantum.cloud.ibm.com/docs/en/api/qiskit-addon-utils/exp-vals#executor_expectation_values) (provided) applies the twirl flips and optional mask and returns one value per (init, twirl).

*Hint: the X-basis values need no other change: the readout rotation already happened on hardware, so `param_index=1` with the same `Z` label gives $\langle X_i\rangle$.*

<div class="alert alert-block alert-success">

✏️ **Your task.** Complete `zstring` and the first two steps of `get_vals`.

</div>

In [ ]:
from qiskit_addon_utils import exp_vals
from qiskit.quantum_info import SparsePauliOp

def zstring(idxs, nbits):
    # Return a Pauli label with 'Z' on the given qubit indices and 'I' elsewhere (little-endian). 

    '''
    TODO: Pauli label of length nbits with 'Z' on the qubits in idxs and 'I' elsewhere;
          qubit 0 is the rightmost character (e.g. zstring([0, 2], 4) == "IZIZ")
    '''
    return ...

def get_vals(result,      # job result
             param_index, # selects the basis: 0 -> Z-basis shots, 1 -> X-basis shots
             idxs,        # indices of the Pauli Z operators for the particular expectation value
             nbits,       # total number of measurement bits (= NS + NE)
             mask=None,   # post-selection mask (used later)
             ):
    """<Z...Z> on the qubits in `idxs`, from one job result, one measurement basis.

    The `meas` register has axes (init, twirl, basis, shots, bits). We slice out the chosen
    basis and hand the (init, twirl, shots, bits) block to `exp_vals`, which applies the
    measurement-flip bookkeeping and (optionally) a post-selection mask, reduces the shots
    axis, and returns one expectation value per (init, twirl) sample.

    param_index selects the basis: 0 -> Z-basis shots, 1 -> X-basis shots.
    """
    
    bool_array = ...  # TODO: step 1 - result["meas"] at basis param_index (axes init, twirl, shots, bits remain)
    sp_op = ...  # TODO: step 2a - SparsePauliOp for the label zstring(idxs, nbits)
    basis_dict = ...  # TODO: step 2b - the dictionary returned by exp_vals.get_measurement_bases(sp_op)
    return np.array(exp_vals.executor_expectation_values(
        bool_array,
        basis_dict,
        measurement_flips=result["measurement_flips.meas"][:, :, param_index, :, :], # this takes care of twirling
        postselect_mask=None if mask is None else mask[:, :, param_index, :],
    ))[0][0]


<div class="alert alert-block alert-success">

✏️ **Your task.** Fill in the function `compute_observables` in the cell below to compute the magnetization and energy density for a given set of expectation values `Zi`, `ZZij`, `Xi`, and System size `NS`.

</div>

In [ ]:
def compute_observables(Zi,   # list of <Z_i> for each System qubit i
                        ZZij, # list of <Z_i Z_j> for each System coupling edge <i,j> on the kagome lattice
                        Xi,   # list of <X_i> for each System qubit i
                        NS,   # number of system qubits
                        ): 
    
    M_Z = ...  # TODO: average <Z_i> over the System qubits
    M_X = ...  # TODO: average <X_i> over the System qubits
    E_over_N = ...  # TODO: energy density E/N_S from <Z_i Z_j>, <X_i> and <Z_i>
    return M_Z, M_X, E_over_N

<div class="alert alert-block alert-info">

**Check your implementation below (5 pts total).**

</div>

In [ ]:
grade_lab_thermal_states_ex2_11(get_vals, compute_observables)

### 2.12 Walk through one configuration

Before sweeping everything, let's unroll the calculation **once** so the mechanics are visible. We take the coldest, deepest run ($\beta=100$, $N_{\rm resets}=7$), its first randomization, and the physical-noise slice, and then build the per-site $\langle Z_i\rangle$, $\langle X_i\rangle$ and per-bond $\langle Z_iZ_j\rangle$ arrays. Averaging over the initializations and twirls gives the expectation values, from which the magnetizations and the energy follow directly.

Run the cell below to compute the observables for this single $(\beta, N_{\rm resets})$ point.

In [ ]:
if part2_results:
    demo_key = "beta_100.0_resets_7"
    demo = part2_results[demo_key]
    random_keys = sorted(demo)          # 'random_0' ... 'random_7'
    kk = 0                              # physical-noise slice (noise factor = 1)

    r0 = demo[random_keys[0]][kk]
    # get_vals returns one value per (init, twirl); shapes below are (site/bond, N_INIT, N_TWIRLS).
    z_site  = np.array([get_vals(r0, 0, [j], nbits) for j in range(NS)])
    x_site  = np.array([get_vals(r0, 1, [j], nbits) for j in range(NS)])
    zz_bond = np.array([get_vals(r0, 0, e, nbits) for e in edge_indices])
    print("shapes -> z_site:", z_site.shape, " x_site:", x_site.shape, " zz_bond:", zz_bond.shape)

    # Average over the initializations and twirls to get this randomization's expectation values.
    Zi = z_site.mean(axis=(1, 2))
    Xi = x_site.mean(axis=(1, 2))
    ZZij = zz_bond.mean(axis=(1, 2))

    # Compute the magnetizations M_Z = 1/NS \sum_i_{NS} <Z_i_{NS}>, M_X = 1/NS \sum_i_{NS} <X_i_{NS}>, and energy density <H_S>/NS.
    M_Z, M_X, E_over_N = compute_observables(Zi, ZZij, Xi, NS)

    print(f"\none randomization of {demo_key} (physical noise):")
    print(f"  M_Z   = {M_Z:+.4f}")
    print(f"  M_X   = {M_X:+.4f}")
    print(f"  E/NS = {E_over_N:+.4f}")
else:
    print("No retrieved results yet -> run the retrieve cell with CONNECT_TO_QPU=True.")

### 2.13 Sweep all configurations

Now we repeat the calculation for every $(\beta, N_{\rm resets})$ thermalization configuration. For each one we compute the energy **per randomization** (averaging over initial states and twirls first), then take the mean across randomizations and use their spread as the standard error.

The energy is linear in the expectation values, so the order of averaging does not change the result.

In [ ]:
from qiskit_addon_utils.noise_management.post_selection import PostSelector

def energy_density(res, mask, NS, edge_indices, nbits):
    # One randomization's E/NS from <Z>, <X> (per site) and <ZZ> (per bond).
    Zi = np.array([get_vals(res, 0, [j], nbits, mask).mean() for j in range(NS)])
    Xi = np.array([get_vals(res, 1, [j], nbits, mask).mean() for j in range(NS)])
    ZZij = np.array([get_vals(res, 0, e,  nbits, mask).mean() for e in edge_indices])

    _, _, E_over_N = compute_observables(Zi, ZZij, Xi, NS)
    return E_over_N

def energy_from_results(results, Lx, Ly, NS, edge_indices, nbits, BETAS, RESETS, NOISE_AMP=None, post_select=False, ALPHA=alpha, NTROT=nTrot, T_TOT=t_tot, GX=gx, GZ=gz): 
    energies_dict = {}
    if not isinstance(BETAS, (list, tuple)):
        BETAS = [BETAS]
    if not isinstance(RESETS, (list, tuple)):
        RESETS = [RESETS]
    if results:
        for beta in BETAS:
            for nreset in RESETS:
                key = f"beta_{beta}_resets_{nreset}"
                key_results = results[key]
                random_keys = sorted(key_results)
                nrandom = len(random_keys)
                if NOISE_AMP is None: 
                    p_factors = [0.0]
                else: 
                    p_factors = NOISE_AMP
                noise_factors = 1 + 2 * np.array(p_factors) 

                if post_select: 
                    base_seed = BASE_SEED + 100 * nreset + int(beta)
                    templates, _, _ = generate_gibbs_circuit(
                        backend, Lx, Ly, nreset, init_environment, H_tot_trot, hardware_layouts, 
                        ALPHA=ALPHA, NTROT=NTROT, T_TOT=T_TOT, GX=GX, GZ=GZ, 
                        beta=beta, noise=p_factors, seed=base_seed, init_system=init_system)
                    post_selector = PostSelector.from_circuit(templates[0], backend.coupling_map)

                energy = np.empty(len(noise_factors))
                energy_sem = np.empty(len(noise_factors))
                energy_per_random = np.empty((nrandom, len(noise_factors)))
                
                for kk, nf in enumerate(noise_factors):
                    per_random = np.empty(nrandom)
                    mask_means = []
                    for rr, rk in enumerate(random_keys):
                        res = key_results[rk][kk]
                        if post_select:
                            mask = post_selector.compute_mask(res, strategy="edge") 
                            mask_means.append(mask.mean()) 
                        else: 
                            mask = None 
                        per_random[rr] = energy_density(res, mask, NS, edge_indices, nbits)

                    mean = per_random.mean()
                    sem  = per_random.std(ddof=1) / np.sqrt(nrandom)
                    
                    energy[kk] = mean
                    energy_sem[kk] = sem
                    energy_per_random[:, kk] = per_random
                    
                    if NOISE_AMP is None: 
                        print(f"beta={beta:6g}  N_resets={nreset}:  E/NS = {mean:+.4f} +/- {sem:.4f}")
                    else: 
                        print(f"beta={beta:6g}  N_resets={nreset}  lambda={nf}:  E/NS = {mean:+.4f} +/- {sem:.4f}")

                    if post_select:
                        print(f"  average surviving-shot fraction = {np.mean(mask_means):.3f}")


                energies_dict[(beta, nreset)] = (energy_per_random, energy, energy_sem)

    else:
        print("No retrieved results yet -> nothing to sweep.")
    return energies_dict


part2_energies = energy_from_results(
    part2_results, Lx, Ly, NS, edge_indices, nbits,
    SMALL_BETAS, SMALL_RESETS
)   # (beta, nreset) -> (per-randomization E/NS, mean E/NS, standard error)


### 2.14 Compare to the noiseless reference

Finally, overlay the physical-noise QPU energies on the noiseless dissipative reference from Part 2.2.

<div class="alert alert-block alert-warning"> 

***Why measure the starting point?***  
We initialize the System from the random $\beta=0$ System ensemble, which has $\langle H_S\rangle/N_S=0$ exactly. On hardware, however, a finite number of randomized preparations and shots will generally produce a value close to, but not exactly, zero. Measuring this point keeps theory and experiment separate and exposes that sampling uncertainty directly. Technically, this point is independent of $\beta$, but we repeat the measurement for both bath temperatures for simplicity and to further expose the sampling uncertainty.

</div>


In [ ]:
if part2_energies:
    fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.4), sharey=True)

    for ax, beta in zip(axes, SMALL_BETAS):
        ref = statevector_reference[beta] / NS
        ref_mean = ref.mean(axis=1)
        ref_sem  = ref.std(axis=1, ddof=1) / np.sqrt(ref.shape[1])
        ax.plot(sv_resets, ref_mean, "--", label="noiseless dissipative reference")
        ax.fill_between(sv_resets, ref_mean - ref_sem, ref_mean + ref_sem, alpha=0.15)

        xs = []
        ys = []
        es = []
        for nreset in SMALL_RESETS:
            _, mean, sem = part2_energies[(beta, nreset)]
            xs.append(nreset)
            ys.append(mean.item())
            es.append(sem.item())

        ax.errorbar(xs, ys, yerr=es, fmt="o", capsize=3, label="QPU, physical noise")
        ax.set_title(fr"$\beta={beta}$")
        ax.set_xlabel(r"$N_{\rm resets}$")
        ax.grid(alpha=0.3)

    axes[0].set_ylabel(r"$E/N_S$")
    axes[1].legend()
    fig.tight_layout()
else:
    print("The QPU-vs-reference figure will appear once both the initial baseline and thermalization results are analyzed.")


<div class="alert alert-block alert-info">

**Save your energy density results (5 pts per $\beta$) using the cell below, and then check your score (10 pts total).**

</div>

In [ ]:
if part2_results:
    with open('submitted_results/part2_results.pkl', "wb") as f: 
        pickle.dump(part2_results, f)
    with open('submitted_results/part2_energies.pkl', "wb") as f: 
        pickle.dump(part2_energies, f)
else:
    print("Nothing saved -> retrieve the Part 2 results first (section 2.10).")


In [ ]:
grade_lab_thermal_states_ex2_14(part2_energies)

# **PART 3: Small-Scale Hardware Experiment with Error Mitigation**


Hopefully you saw the same general trend of convergence towards steady state in your first hardware results; however, it is unlikely that the hardware results exactly match the noiseless statevector simulation. Here, we will modify the implementation to employ *error mitigation.* This will be critically important as we scale to the large hardware experiment in the final portion of this challenge.

In this section, we study error mitigation at the deepest small-<span style="color:darkgreen">System</span> point,

$$
N_{\rm resets}=7,
$$

for **both bath temperatures**:

$$
\beta=100,\qquad \beta=0.25.
$$

The section is intentionally organized as a **worked example followed by a repetition**:

- **Walkthrough:** ($\beta=100$). We step through leakage-sensitive post-selection and zero-noise extrapolation in detail.
- **Repeat the workflow:** ($\beta=0.25$). You will apply the same analysis.

### 3.1 Error mitigation and error detection

We use two complementary techniques.

- **Leakage-sensitive post-selection.** Each circuit contains the ordinary data register `meas` and an additional `meas_ps` register generated with the calibrated `xslow` instruction. A `PostSelector` uses these extra checks to construct a keep/discard mask. With the `"edge"` strategy we reject shots that have at least two qubits linked by a hardware edge that fail the leakage-sensitive consistency test. Post-selection does not change the circuit result itself; it changes which shots are retained in the observable estimate. This is a form of *error detection*.

- **Zero-noise extrapolation (ZNE).** We deliberately increase the number of noisy two-qubit-gate applications without changing the logical action of the circuit. With amplification *probability* ($p$), the two-qubit noise-amplification *factor* is

$$
\lambda = 1+2p .
$$

We measure the energy at a minimum of three different noise levels, fit a first-order model $E(\lambda)$, and extrapolate to $\lambda=0$.

Our mitigation chain is:

$$
\text{raw energy} \;\rightarrow\; \text{post-selected energy} \;\rightarrow\; \text{ZNE energy, with and without post-selection}.
$$

*Note:* For valid ZNE, the three noise-amplification variants must represent the **same ideal stochastic realization**. The accompanying `gibbs_circuit_generator` therefore keeps the sampled initial <span style="color:red">Environment</span> state, Bohr frequencies, jump operators, and initial <span style="color:darkgreen">System</span> state fixed across the three $\lambda$ values within a randomization; only the noise amplification changes.


These techniques can only address the error due to noise; a systematic error due to the inherent limited accuracy of the dissipative Gibbs sampling algorithm will remain. This is illustrated in the figure below.
<p align="center">
    <img src="./figures/ZNE.png" width="800">
</p>

### 3.2 Submit the ZNE experiments

We use the same reset depth and ZNE grid for both temperatures:

$$
N_{\rm resets}=7,\qquad \beta = \{0.25, 100\}.
$$

Both temperatures are submitted through the same helper and stored in one job-ID file.

<div class="alert alert-block alert-success">

✏️ **Your task.** Fill in the `ZNE_NOISE_AMP` variable with a list of at least three values for $p$ in the cell below. Be sure to include the point 0.0, which corresponds to the physical noise of the circuit without noise amplification. (Suggested values: 0.0, 0.25, 0.5 -- but feel free to change these!)

</div>

In [ ]:
ZNE_BETAS = (100.0, 0.25)
ZNE_RESET = 7

RUN_QPU_MITIGATION = False  # set to True to submit the ZNE job to the QPU

# Gate-folding probabilities p. The corresponding nominal amplification factors are
# lambda = 1 + 2p
ZNE_NOISE_AMP = [..., ..., ...]  # TODO: at least three values of p, including 0.0
ZNE_FACTORS = 1 + 2 * np.asarray(ZNE_NOISE_AMP)

ZNE_JOB_FILE = Path(
    f"./job_info/qdc26_small_gibbs_zne_{BACKEND_NAME}_betas100_025_resets{ZNE_RESET}.yaml"
)

submit_to_QPU(
    RUN_QPU=RUN_QPU_MITIGATION,
    JOB_FILE=ZNE_JOB_FILE,
    Lx=Lx,
    Ly=Ly,
    NS=NS,
    BETAS=ZNE_BETAS,
    RESETS=ZNE_RESET,
    BASE_SEED=BASE_SEED,
    N_RANDOMIZATIONS=N_RANDOMIZATIONS,
    N_SHOTS=N_SHOTS,
    N_INIT=N_INIT,
    N_TWIRLS=N_TWIRLS,
    NOISE_AMP=ZNE_NOISE_AMP,
)

print("mitigation betas:", ZNE_BETAS)
print("lambda values:", ZNE_FACTORS)


### 3.3 Retrieve the mitigation experiments

Run the cell below to retrieve your results.

In [ ]:
part3_results = retrieve_results(
    service, ZNE_JOB_FILE, CACHE_FILE=Path('./submitted_results/part3_results.pkl')
)

if part3_results:
    for beta in ZNE_BETAS:
        key = f"beta_{beta}_resets_{ZNE_RESET}"
        print(
            f"beta={beta}: retrieved {len(part3_results[key])} randomization(s), "
            f"each containing {len(ZNE_NOISE_AMP)} noise-amplification points."
        )
else:
    print("No ZNE results retrieved yet.")


Now, retrieve the energy using `energy_from_results`, which was defined earlier. We're going to do this twice: once using the raw data, and once using the post-selected shots. This can all be done using `energy_from_results`, which was defined earlier. Take a closer look at the function to see what it does when the extra arguments `post_select` and `NOISE_AMP` aren't set to the default values.

In [ ]:
part3_energies_raw = energy_from_results(part3_results, Lx, Ly, NS, edge_indices, nbits, SMALL_BETAS, ZNE_RESET, post_select=False, NOISE_AMP=ZNE_NOISE_AMP)
part3_energies_ps = energy_from_results(part3_results, Lx, Ly, NS, edge_indices, nbits, SMALL_BETAS, ZNE_RESET, post_select=True, NOISE_AMP=ZNE_NOISE_AMP)

### 3.4 Implement zero-noise extrapolation
We now have the noise-amplified data. Next, we have to implement the extrapolation. We will use a linear fit for this:

$$
\langle E/N \rangle(\lambda) = a+b\lambda,
$$

and use the mean and standard error of the intercept ($a$) as the final zero-noise estimate and uncertainty, respectively.

<div class="alert alert-block alert-success">

✏️ **Your task.** Fill in the `linear_zne` function below to perform this fit, given a series of `noise_factors` and (expectation) `values`. The function should return the zero-noise extrapolated value.

</div>

In [ ]:
def linear_zne(noise_factors, # array of noise amplification factors (lambda)
               values,        # array of corresponding expectation values
               ):
    # perform a linear fit to the noise factors and return the zero-noise extrapolation estimate.
    '''
    TODO: zero-noise intercept of a linear fit of values vs. noise_factors
    '''
    return ...

Now, let's write a function that takes the energy density data and uses `linear_zne` to perform the fit.

<div class="alert alert-block alert-success">

✏️ **Your task.** Fill in the function below to compute `zne_estimates`, which will be an array of length equal to the number of randomizations.

</div>

In [ ]:
def perform_zne(energies,  # energy data as returned from energy_from_results
                NOISE_AMP, # noise amplification probabilities
                BETA,      # inverse temperature
                RESETS,    # n_{resets}
                ):
    """Linear ZNE, fit once per independent randomization."""

    noise_factors = 1 + 2*np.array(NOISE_AMP)

    energy_per_randomizations, _, _ = energies[(BETA, RESETS)]

    zne_estimates = ...  # TODO: array of linear_zne intercepts, one per randomization (row)
    zne_mean = zne_estimates.mean()
    zne_sem = zne_estimates.std(ddof=1) / np.sqrt(len(zne_estimates))

    return {
        "zne": float(zne_mean),
        "zne_err": float(zne_sem),
    }

<div class="alert alert-block alert-info">

**Check your implementation below (10 pts total).**

</div>

In [ ]:
grade_lab_thermal_states_ex3_4(linear_zne, perform_zne)

### 3.5 The cold bath experiment
Let's examine the $\beta = 100$ case with a plot below.

In [ ]:
BETA = 100.0
part3_zne_beta100_raw = perform_zne(part3_energies_raw, ZNE_NOISE_AMP, BETA, ZNE_RESET)
part3_zne_beta100_ps = perform_zne(part3_energies_ps, ZNE_NOISE_AMP, BETA, ZNE_RESET)

_, energy_raw, energy_sem_raw = part3_energies_raw[(BETA,ZNE_RESET)]
_, energy_ps, energy_sem_ps = part3_energies_ps[(BETA,ZNE_RESET)]

lambdas = 1 + 2*np.array(ZNE_NOISE_AMP)
ref = statevector_reference[BETA][ZNE_RESET].mean() / NS

fig, ax = plt.subplots(figsize=(5.2, 4.0), dpi=300)
ax.errorbar(lambdas, energy_raw, yerr=energy_sem_raw, fmt="o", capsize=3, label="raw QPU")
ax.errorbar(lambdas, energy_ps, yerr=energy_sem_ps, fmt="s", capsize=3, label="post-selected QPU")
ax.errorbar([0], [part3_zne_beta100_raw["zne"]], yerr=[part3_zne_beta100_raw["zne_err"]], fmt="D", mfc="none", capsize=3, label="raw ZNE estimate")
ax.errorbar([0], [part3_zne_beta100_ps["zne"]], yerr=[part3_zne_beta100_ps["zne_err"]], fmt="D", mfc="none", capsize=3, label="post-selected ZNE estimate")
ax.axhline(ref, ls="--", lw=1, color="k", label="noiseless dissipative reference")
ax.set_xlabel(r"nominal noise factor $\lambda$")
ax.set_ylabel(r"$E/N_S$")
ax.set_title(fr"$\beta={BETA},\ N_{{\rm resets}}={ZNE_RESET}$")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()

The noiseless dissipative result is the appropriate benchmark for the mitigation procedure because it uses the same finite-depth preparation protocol with hardware noise removed.

The exact Gibbs energy is a different target. Any residual gap

$$
E_{\rm noiseless\ protocol}-E_{\rm exact\ Gibbs}
$$

is an algorithmic / fixed-point error of the approximate preparation channel, not a hardware error that post-selection or ZNE should remove. Let's add this comparison to the plot, so you can visualize the gap. 

In [ ]:
H_mat = np.asarray(model.hamiltonian.to_matrix())
evals = np.linalg.eigvalsh(H_mat)

def exact_thermal_density(beta):
    w = np.exp(-beta * (evals - evals.min()))
    return float((evals * w).sum() / w.sum()) / NS

thermal_ref = exact_thermal_density(BETA)
fig, ax = plt.subplots(figsize=(5.2, 4.0), dpi=300)
ax.errorbar(lambdas, energy_raw, yerr=energy_sem_raw, fmt="o", capsize=3, label="raw QPU")
ax.errorbar(lambdas, energy_ps, yerr=energy_sem_ps, fmt="s", capsize=3, label="post-selected QPU")
ax.errorbar([0], [part3_zne_beta100_raw["zne"]], yerr=[part3_zne_beta100_raw["zne_err"]], fmt="D", mfc="none", capsize=3, label="raw ZNE estimate")
ax.errorbar([0], [part3_zne_beta100_ps["zne"]], yerr=[part3_zne_beta100_ps["zne_err"]], fmt="D", mfc="none", capsize=3, label="post-selected ZNE estimate")
ax.axhline(ref, ls="--", lw=1, color="k", label="noiseless dissipative reference")
ax.axhline(thermal_ref, ls="--", lw=1, color="r", label="exact reference")
ax.set_xlabel(r"nominal noise factor $\lambda$")
ax.set_ylabel(r"$E/N_S$")
ax.set_title(fr"$\beta={BETA},\ N_{{\rm resets}}={ZNE_RESET}$")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()

<div class="alert alert-block alert-info">

**Save your energy density results using the cell below, and then check your score (5 pts).**

</div>

In [ ]:
if part3_results:
    with open('submitted_results/part3_results.pkl', "wb") as f: 
        pickle.dump(part3_results, f)
    with open('submitted_results/part3_energies_raw.pkl', "wb") as f: 
        pickle.dump(part3_energies_raw, f)
    with open('submitted_results/part3_energies_ps.pkl', "wb") as f: 
        pickle.dump(part3_energies_ps, f)
    with open('submitted_results/part3_zne_beta100_raw.pkl', "wb") as f: 
        pickle.dump(part3_zne_beta100_raw, f)
    with open('submitted_results/part3_zne_beta100_ps.pkl', "wb") as f: 
        pickle.dump(part3_zne_beta100_ps, f)
else:
    print("Nothing saved -> retrieve the Part 3 results first (section 3.3).")


In [ ]:
grade_lab_thermal_states_ex3_5(part3_zne_beta100_ps)

### 3.6 The warm bath experiment

<div class="alert alert-block alert-success">

✏️ **Your task.** Repeat the analysis for $\beta = 0.25$ in the cell below:

- Compute the raw and post-selected ZNE estimates.
- Create a plot like the one above for the $\beta = 0.25$ case.

</div>

In [ ]:
BETA = 0.25
part3_zne_beta0p25_raw = ...  # TODO: perform_zne on the raw energies at this BETA
part3_zne_beta0p25_ps = ...  # TODO: perform_zne on the post-selected energies at this BETA

# TODO: plot the raw, post-selected and ZNE energies with both references, as in Part 3.5

<div class="alert alert-block alert-info">

**Save your energy density results using the cell below, and then check your score (5 pts).**

</div>

In [ ]:
if part3_results:
    with open('submitted_results/part3_zne_beta0p25_raw.pkl', "wb") as f: 
        pickle.dump(part3_zne_beta0p25_raw, f)
    with open('submitted_results/part3_zne_beta0p25_ps.pkl', "wb") as f: 
        pickle.dump(part3_zne_beta0p25_ps, f)
else:
    print("Nothing saved -> retrieve the Part 3 results first (section 3.3).")


In [ ]:
grade_lab_thermal_states_ex3_6(part3_zne_beta0p25_ps)

# **PART 4: Large-Scale Hardware Experiment — 67 Active Qubits**


Part 4 takes the same dissipative-preparation and mitigation workflow to a much larger hardware experiment:

$$
(L_x, L_y) = (3,3), \qquad N_S=36,\qquad N_E=31,\qquad N_{\rm tot}=67.
$$

For your convenience, the virtual qubit layout for the large-scale <span style="color:darkgreen">System</span> is provided again here. 
<p align="center">
    <img src="./figures/large-scale_virtual-layout.png" width="550">
</p>


### 4.1 Scaling the lattice

To scale to this larger lattice, we need to add this lattice case to the previous lattice-dependent helper functions: `hardware_layouts`, `kagome_edge_set`, and `SE_Coupling`.

<div class="alert alert-block alert-success">

✏️ **Your task.** You'll need to "place" the lattice onto physical qubits. One possible mapping is provided in the `figures` folder, but please translate the lattice if needed to avoid noisy qubits. The functions are provided below; please add a case for $(L_x, L_y) = (3, 3)$ to each function. Note that now that $N_E < N_S$, the `SE_Coupling` function should return one sampled pairing of length $N_E$. Use `np.random.choice` to pick between two <span style="color:darkgreen">System</span> qubits, when applicable.

</div>

In [ ]:
def hardware_layouts(Lx, Ly):
    # convention: [system qubits, left to right, top to bottom] + [environment qubits in the order that they are paired with the ordered system qubits]
    layout_dict = {(2,2):[38,39,48,50,52,54,57,58,66,68,70,72] + [29,33,49,51,53,55,47,71,65,67,69,73],
                   (3,3):[] + [],  # TODO: 36 System + 31 Environment physical qubits
                   }
    return layout_dict[(Lx,Ly)]


def kagome_edge_set(Lx, Ly):
    # Returns the System-system ZZ bonds used in H_S and in the energy estimator.
    edge_set_dict = {
        (2, 2): [(0,2),(0,3),(1,4),(1,5),(2,3),(3,4),(4,5),(2,6),(3,7),
                 (4,7),(6,8),(6,9),(7,10),(7,11),(8,9),(9,10),(10,11)],

        (3, 3): [],  # TODO: the 59 System ZZ bonds (i, j) of the (3,3) lattice
    }
    return edge_set_dict[(Lx, Ly)]


def SE_Coupling(Lx, Ly):
    # returns a list of system qubits -- one for each environment qubit -- to establish the system-environment couplings for a particular application of Phi (virtual indices, not the hardware layout).
    coupling_dict = {
        (2, 2): np.arange(12),

        (3, 3): [],  # TODO: one System partner per Environment qubit (31); np.random.choice where two are possible
    }
    return coupling_dict[(Lx, Ly)]


With the `(3,3)` cases filled in above, the same helpers now describe the **large** lattice: **36 <span style="color:darkgreen">System</span> qubits + 31 <span style="color:red">Environment</span> qubits = 67 physical qubits**. We wire the geometry into the analysis like we did at small scale (Part 2.5).

In [ ]:
# Internal routing key for the 36-site large patch.
Lx_L, Ly_L = 3, 3

NS_L = 36
layout_L = hardware_layouts(Lx_L, Ly_L)
system_qubits_L = layout_L[:NS_L]
env_qubits_L = layout_L[NS_L:]

NE_L = len(env_qubits_L)
nbits_L = NS_L + NE_L

edge_indices_L = kagome_edge_set(Lx_L, Ly_L)
Nbonds_L = len(edge_indices_L)

assert NS_L == 36
assert NE_L == 31
assert nbits_L == 67
assert max(max(e) for e in edge_indices_L) < NS_L
assert len(edge_indices_L) == 59

print(f"System qubits      : {NS_L}")
print(f"Environment qubits : {NE_L}")
print(f"Total qubits       : {nbits_L}")
print(f"System ZZ bonds    : {Nbonds_L}")


<div class="alert alert-block alert-info">

**Check your implementation below (10 pts total).**

</div>

In [ ]:
grade_lab_thermal_states_ex4_1(kagome_edge_set, SE_Coupling)

### 4.2 Build one representative 67-qubit circuit

Before spending QPU time, build a single physical-noise template and inspect its resource footprint. This is a useful sanity check that the intended large routing key is actually being used. *Note that despite the larger size of the lattice, the 2-qubit depth should be <span style="color:darkgreen">System</span>-size independent thanks to the highly-optimized Trotterization implementation.*


In [ ]:
large_demo_templates = None

if backend is not None:
    large_demo_templates, large_demo_samplexes, _ = generate_gibbs_circuit(
        backend,
        Lx_L,
        Ly_L,
        7,
        init_environment,
        H_tot_trot,
        hardware_layouts,
        ALPHA=alpha, NTROT=nTrot, T_TOT=t_tot, GX=gx, GZ=gz, 
        beta=100.0,
        seed=BASE_SEED,
        init_system=init_system,
    )

    circ = large_demo_templates[0]
    part4_two_qubit_depth = circ.depth(
        filter_function=lambda inst: inst.operation.name in ("cz", "rzz")
    )
    reset_count = circ.count_ops().get("reset", 0)

    print("templates returned             :", len(large_demo_templates))
    print("two-qubit (cz/rzz) depth       :", part4_two_qubit_depth)
    print("reset operations               :", reset_count)
    print("dynamical-decoupling X pulses  :", circ.count_ops().get("x", 0))
    print("classical registers            :", [creg.name for creg in circ.cregs])
else:
    print("Backend not connected -> skipping circuit build.")

<div class="alert alert-block alert-info">

**Check your 2-qubit gate depth below (5 pts).**

</div>

In [ ]:
grade_lab_thermal_states_ex4_2(part4_two_qubit_depth, part2_two_qubit_depth)

### 4.3 Submit the large-scale experiment

We now run a single cold, deep preparation,

$$
\beta=100,\qquad N_{\rm resets}=7.
$$

We'll want to use ZNE and post-selection like in Part 3.

The cell is safe-guarded by the `RUN_QPU_LARGE` boolean.

<div class="alert alert-block alert-success">

✏️ **Your task.** Select a minimum of three noise amplification probabilities (include the value 0.0) for the variable `LARGE_NOISE_AMP` below.

</div>

In [ ]:
LARGE_BETA = 100.0
LARGE_QPU_RESET = 7
LARGE_NOISE_AMP = [..., ..., ...]  # TODO: at least three values of p, including 0.0
LARGE_FACTORS = 1 + 2 * np.asarray(LARGE_NOISE_AMP)

LARGE_JOB_FILE = Path(
    f"./job_info/qdc26_large_gibbs_job_ids_{BACKEND_NAME}_beta{LARGE_BETA}.yaml"
)

RUN_QPU_LARGE = False  # Set to False to skip the large-scale QPU submission.

# Start with the same sampling settings used in Part 2.
# These can be reduced after profiling on the workshop backend if needed.
N_RAND_L = N_RANDOMIZATIONS
N_INIT_L = N_INIT
N_TWIRLS_L = N_TWIRLS
N_SHOTS_L = N_SHOTS

samplex_points_per_noise = N_INIT_L * N_TWIRLS_L * 2
requested_shots = (
    N_RAND_L
    * len(LARGE_NOISE_AMP)
    * samplex_points_per_noise
    * N_SHOTS_L
)

print("Large-scale QPU checkpoint")
print("--------------------------")
print(f"active qubits                  : {nbits_L}")
print(f"beta                           : {LARGE_BETA}")
print(f"N_resets                       : {LARGE_QPU_RESET}")
print(f"nominal ZNE factors            : {LARGE_FACTORS.tolist()}")
print(f"circuit randomizations         : {N_RAND_L}")
print(f"initial states / randomization : {N_INIT_L}")
print(f"twirls / initial state         : {N_TWIRLS_L}")
print(f"shots / sweep point            : {N_SHOTS_L}")
print(f"requested measurement shots    : {requested_shots:,}")

submit_to_QPU(
    RUN_QPU=RUN_QPU_LARGE,
    JOB_FILE=LARGE_JOB_FILE,
    Lx=Lx_L, 
    Ly=Ly_L,
    NS=NS_L,
    BETAS=LARGE_BETA,
    RESETS=LARGE_QPU_RESET,
    BASE_SEED=BASE_SEED,
    N_RANDOMIZATIONS=N_RAND_L,
    N_SHOTS=N_SHOTS_L,
    N_INIT=N_INIT_L,
    N_TWIRLS=N_TWIRLS_L,
    NOISE_AMP=LARGE_NOISE_AMP,
)


### 4.4 Retrieve the 67-qubit results

Now, retrieve the job results. As before, retrieval only reads previously submitted jobs. If `submitted_results/part4_results.pkl` exists and is non-empty, it is loaded and no backend connection is needed. Otherwise this requires `CONNECT_TO_QPU = True`: the cell checks the status of every job, and if any job is still queued or running, it prints a status summary and returns without waiting; rerun it once all jobs are complete.


In [ ]:
part4_results = retrieve_results(
    service, LARGE_JOB_FILE, CACHE_FILE=Path('./submitted_results/part4_results.pkl')
)

### 4.5 Reuse the mitigation workflow at large scale

There is no new mitigation idea in Part 3 — we run the exact same chain as at small scale (Part 3.1):

$$
\text{raw energy} \;\rightarrow\; \text{post-selected energy} \;\rightarrow\; \text{ZNE energy, with and without post-selection}.
$$

We can reuse the function we already wrote to do this. With `post_select=True`, `energy_from_results` prints the surviving-shot fraction for each noise factor. Expect a lower survival fraction for the larger experiment (with more qubits involved, it is more likely that a shot will be corrupted by leakage).

<div class="alert alert-block alert-success">

✏️ **Your task.** Call `energy_from_results` twice — once on the raw shots (`post_select=False`) and once on the post-selected shots (`post_select=True`) — passing the **large-scale** geometry and the ZNE grid `LARGE_NOISE_AMP`.

</div>

In [ ]:
part4_energies_raw = None
part4_energies_ps = None

if part4_results and backend is not None:
    # Reuse energy_from_results (defined in Part 2.13) at the large-scale geometry.
    part4_energies_raw = ...  # TODO: energy_from_results on part4_results with the large-scale geometry, post_select=False
    part4_energies_ps = ...  # TODO: the same with post_select=True
else:
    print("Retrieve the large-scale results and connect to the backend first.")

### 4.6 Extrapolate to zero noise

As in Part 3, the ZNE intercept is computed **once per independent circuit randomization** and then averaged, so the uncertainty comes from the spread across randomizations rather than from a single line fit.

<div class="alert alert-block alert-success">

✏️ **Your task.** Reuse `perform_zne` (Part 3.4) to fit the series of noise-amplified expectation values, for both the raw and the post-selected series.

</div>

In [ ]:
part4_zne_raw = None
part4_zne_ps = None

if part4_energies_raw and part4_energies_ps is not None:
    # Reuse perform_zne (defined in Part 3.4): one linear fit per randomization, then averaged.
    part4_zne_raw = ...  # TODO: perform_zne on part4_energies_raw
    part4_zne_ps = ...  # TODO: perform_zne on part4_energies_ps

    # The physical-noise point is the first noise factor (p = 0.0 -> lambda = 1).
    _, energy_raw, energy_sem_raw = part4_energies_raw[(LARGE_BETA, LARGE_QPU_RESET)]
    _, energy_ps, energy_sem_ps = part4_energies_ps[(LARGE_BETA, LARGE_QPU_RESET)]

    print(f"raw physical-noise energy      : {energy_raw[0]:+.4f} +/- {energy_sem_raw[0]:.4f}")
    print(f"post-selected physical energy  : {energy_ps[0]:+.4f} +/- {energy_sem_ps[0]:.4f}")
    print(f"post-selected ZNE estimate     : {part4_zne_ps['zne']:+.4f} +/- {part4_zne_ps['zne_err']:.4f}")
else:
    print("No large-scale mitigation data available yet.")

### 4.7 Visualize the large-scale mitigation

Finally, plot the raw and post-selected energies against the noise factor $\lambda$, together with the raw and post-selected ZNE estimates at $\lambda=0$ — the same picture we drew at small scale (Part 3.5).

At small scale we compared against a noiseless dissipative reference and the exact Gibbs energy. Neither is available here: exact diagonalization of a 36-spin Hamiltonian ($2^{36}$ states) is not feasible, and neither is a statevector simulation of the algorithm. Instead we compare against a reference value computed with the quantum Monte Carlo (QMC) algorithm. Note that this reference value is an estimate of the exact thermal value, not the "noiseless simulation." We can no longer visualize the two error gaps (hardware noise and systematic error due to the algorithm) separately, so you may see a pretty large disagreement between your results and the reference value. 

In [ ]:
LARGE_REFERENCE = -1.4

if part4_zne_raw is not None and part4_zne_ps is not None:
    lambdas = 1 + 2 * np.array(LARGE_NOISE_AMP)

    _, energy_raw, energy_sem_raw = part4_energies_raw[(LARGE_BETA, LARGE_QPU_RESET)]
    _, energy_ps, energy_sem_ps = part4_energies_ps[(LARGE_BETA, LARGE_QPU_RESET)]

    fig, ax = plt.subplots(figsize=(5.2, 4.0), dpi=300)
    ax.errorbar(lambdas, energy_raw, yerr=energy_sem_raw, fmt="o", capsize=3, label="raw QPU")
    ax.errorbar(lambdas, energy_ps, yerr=energy_sem_ps, fmt="s", capsize=3, label="post-selected QPU")
    ax.errorbar([0], [part4_zne_raw["zne"]], yerr=[part4_zne_raw["zne_err"]], fmt="D", mfc="none", capsize=3, label="raw ZNE estimate")
    ax.errorbar([0], [part4_zne_ps["zne"]], yerr=[part4_zne_ps["zne_err"]], fmt="D", mfc="none", capsize=3, label="post-selected ZNE estimate")
    if LARGE_REFERENCE is not None:
        ax.axhline(LARGE_REFERENCE, ls="--", lw=1, color="r", label="verified reference")
    ax.set_xlabel(r"nominal noise factor $\lambda$")
    ax.set_ylabel(r"$E/N_S$")
    ax.set_title(fr"67 active qubits: $\beta={LARGE_BETA},\ N_{{\rm resets}}={LARGE_QPU_RESET}$")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="center right")
    fig.tight_layout()
else:
    print("The large-scale mitigation figure will appear once the results are analyzed.")

<div class="alert alert-block alert-info">

**Part 3 takeaway**

The large-scale experiment uses **67 actively participating qubits** and the same physics-and-mitigation workflow developed at small scale. The key difference is the verification regime: full statevector benchmarking is no longer practical, so the experiment is assessed through physically meaningful observables and internal consistency of the mitigation procedure.

</div>


<div class="alert alert-block alert-info">

**Save your energy density results using the cell below, and then check your score (15 pts).**

</div>

In [ ]:
if part4_results:
    with open('submitted_results/part4_results.pkl', "wb") as f: 
        pickle.dump(part4_results, f)
    with open('submitted_results/part4_energies_raw.pkl', "wb") as f: 
        pickle.dump(part4_energies_raw, f)
    with open('submitted_results/part4_energies_ps.pkl', "wb") as f: 
        pickle.dump(part4_energies_ps, f)
    with open('submitted_results/part4_zne_raw.pkl', "wb") as f: 
        pickle.dump(part4_zne_raw, f)
    with open('submitted_results/part4_zne_ps.pkl', "wb") as f: 
        pickle.dump(part4_zne_ps, f)
else:
    print("Nothing saved -> retrieve the Part 4 results first (section 4.4).")


In [ ]:
grade_lab_thermal_states_ex4_7(part4_zne_ps)

<div class="alert alert-block alert-success">

**Congratulations on making it this far!** 🎉

</div>

<div class="alert alert-block alert-warning"> 

**Bonus: Improving your hardware results**

To improve your results, consider changing the algorithm parameters (number of randomizations, number of twirls, etc.). You could also try a different number of noise amplification probabilities for ZNE, or even a different extrapolation strategy, if your data does not follow a linear trend. 

If desired, repeat the ZNE + post-selection on other values of $N_{\rm resets}$. You could also consider trying a value of $N_{\rm resets} > 7$ to see how the steady state persists. 

If you change your implementation and generate better results, please overwrite your "best" submission files and submit the "best" versions to the grader. 
</div>

In [ ]:
# By default, "best" results / energies are saved as the same. 
# If you change your implementation and improve your results, please overwrite the "best" energy files. 
part2_best_results = part2_results
part2_best_energies = part2_energies
if part2_best_results:
    with open('submitted_results/part2_best_results.pkl', "wb") as f: 
        pickle.dump(part2_best_results, f)
    with open('submitted_results/part2_best_energies.pkl', "wb") as f: 
        pickle.dump(part2_best_energies, f)
else:
    print("Nothing saved -> retrieve the Part 2 results first (section 2.10).")


In [ ]:
grade_lab_thermal_states_ex2_14(part2_best_energies)

In [ ]:
# By default, "best" results / energies are saved as the same. 
# If you change your implementation and improve your results, please overwrite the "best" energy files. 
part3_best_results = part3_results
part3_best_energies_raw = part3_energies_raw
part3_best_energies_ps = part3_energies_ps
part3_best_zne_beta100_raw = part3_zne_beta100_raw
part3_best_zne_beta100_ps = part3_zne_beta100_ps
if part3_best_results:
    with open('submitted_results/part3_best_results.pkl', "wb") as f: 
        pickle.dump(part3_best_results, f)
    with open('submitted_results/part3_best_energies_raw.pkl', "wb") as f: 
        pickle.dump(part3_best_energies_raw, f)
    with open('submitted_results/part3_best_energies_ps.pkl', "wb") as f: 
        pickle.dump(part3_best_energies_ps, f)
    with open('submitted_results/part3_best_zne_beta100_raw.pkl', "wb") as f: 
        pickle.dump(part3_best_zne_beta100_raw, f)
    with open('submitted_results/part3_best_zne_beta100_ps.pkl', "wb") as f: 
        pickle.dump(part3_best_zne_beta100_ps, f)
else:
    print("Nothing saved -> retrieve the Part 3 results first (section 3.3).")


In [ ]:
grade_lab_thermal_states_ex3_5(part3_best_zne_beta100_ps)

In [ ]:
# By default, "best" results / energies are saved as the same. 
# If you change your implementation and improve your results, please overwrite the "best" energy files. 
part3_best_zne_beta0p25_raw = part3_zne_beta0p25_raw
part3_best_zne_beta0p25_ps = part3_zne_beta0p25_ps
if part3_results:
    with open('submitted_results/part3_best_zne_beta0p25_raw.pkl', "wb") as f: 
        pickle.dump(part3_best_zne_beta0p25_raw, f)
    with open('submitted_results/part3_best_zne_beta0p25_ps.pkl', "wb") as f: 
        pickle.dump(part3_best_zne_beta0p25_ps, f)
else:
    print("Nothing saved -> retrieve the Part 3 results first (section 3.3).")


In [ ]:
grade_lab_thermal_states_ex3_6(part3_best_zne_beta0p25_ps)

In [ ]:
# By default, "best" results / energies are saved as the same. 
# If you change your implementation and improve your results, please overwrite the "best" energy files. 
part4_best_results = part4_results
part4_best_energies_raw = part4_energies_raw
part4_best_energies_ps = part4_energies_ps
part4_best_zne_raw = part4_zne_raw
part4_best_zne_ps = part4_zne_ps
if part4_best_results:
    with open('submitted_results/part4_best_results.pkl', "wb") as f: 
        pickle.dump(part4_best_results, f)
    with open('submitted_results/part4_best_energies_raw.pkl', "wb") as f: 
        pickle.dump(part4_best_energies_raw, f)
    with open('submitted_results/part4_best_energies_ps.pkl', "wb") as f: 
        pickle.dump(part4_best_energies_ps, f)
    with open('submitted_results/part4_best_zne_raw.pkl', "wb") as f: 
        pickle.dump(part4_best_zne_raw, f)
    with open('submitted_results/part4_best_zne_ps.pkl', "wb") as f: 
        pickle.dump(part4_best_zne_ps, f)
else:
    print("Nothing saved -> retrieve the Part 4 results first (section 4.4).")


In [ ]:
grade_lab_thermal_states_ex4_7(part4_best_zne_ps)

© IBM Corp. 2026